<a href="https://colab.research.google.com/github/rija-ansari/MSE1003H_RijaAnsari/blob/main/Final_Project/MSE1003_Final_Project_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Comment out the below cells when running in Colab


In [ ]:
#!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/

In [ ]:
import os
#repo = "MSE1003H_RijaAnsari"
#print(os.listdir(repo))


In [ ]:
#%cd MSE1003H_RijaAnsari/Final_Project

# MSE1003: Final Project: Community Science and Self-Driving Labs

This project simulates the role of a staff scientist working to take a 3D printing self-driving lab (SDL) from post-commission to online, available for use by the scientific community.

To pilot this, 13 staff scientists will provide proposals in order to test a proposal-selection acquistion policy that bears in mind the physical limitations of the SDL at around 20+ samples daily. 

**Goal: For a set of proposed 3D printing experiments from a scientific community, develop an
acquisition policy that maximizes (1) scientific discovery and (2) community engagement.**

## Introduction

**Scientific Discovery:**

According to the Stanford Encyclopedia of Philosophy, "scientific discovery is the process or product of successful scientific inquiry. Objects of discovery can be things, events, processes, causes and properties as well as theories and hypotheses and their features." [1]

For the purposes of this project, scientific discovery will be defined as experiments that explore novel areas in the feature space, have a high predicted target variable y and have high-model uncertainty. This will be quantified with large distance from current experiments, a strong energy absorption and most learning from the upper confidence bounded points.

**Community Engagement:**

Community engagemnent is defined as "seeks to better engage the community to achieve long-term and sustainable outcomes, processes, relationships, discourse, decision-making or implementation. To be successful, it must encompass strategies and processes that are senstive to the community context in which it occurs." [2]

For this project, experiments that engage the community should be interpretable, integrate vast community interests, and have a simple and justifiable explanation. 

[1] https://plato.stanford.edu/entries/scientific-discovery/

[2] https://aese.psu.edu/research/centers/cecd/engagement-toolbox/engagement/what-is-community-engagement


## Designing the approach

Since this project is a simulated approach, a subset of data will be used to train the surrogate model and the remainder of the data will be "unseen" and iteratively acquired from to represent new experiments from the SDL. 

In [ ]:
import os
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error, max_error, mean_absolute_percentage_error

from sklearn.gaussian_process import GaussianProcessRegressor 
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic, ConstantKernel


from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

from scipy.stats import norm


In [ ]:
#install botorch quietly
!pip install botorch -q

In [ ]:
# ─── Run this first to check your BoTorch version ─────────────────────────────
import botorch, gpytorch, torch
print(f"BoTorch:  {botorch.__version__}")
print(f"GPyTorch: {gpytorch.__version__}")
print(f"PyTorch:  {torch.__version__}")

In [ ]:
import torch
import gpytorch
from gpytorch.means import ConstantMean, MultitaskMean
from gpytorch.kernels import RBFKernel, ScaleKernel, IndexKernel, MultitaskKernel
from gpytorch.distributions import MultitaskMultivariateNormal

from gpytorch.models import ExactGP
from gpytorch.likelihoods import MultitaskGaussianLikelihood
from gpytorch.distributions import MultitaskMultivariateNormal
from gpytorch.means import MultitaskMean, ConstantMean
from gpytorch.kernels import ScaleKernel, RBFKernel, MultitaskKernel

from botorch.models.model import Model
from botorch.posteriors.gpytorch import GPyTorchPosterior
from botorch.acquisition.objective import GenericMCObjective
from botorch.acquisition.monte_carlo import qNoisyExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.optim import optimize_acqf

In [ ]:
data = pd.read_csv("Data.csv")

In [ ]:
data.head()

In [ ]:
df = data.copy()

In [ ]:
df.shape

The full data has a range of 13224 experiments, with the pool index, 11 features (x#) and the target variable y. Feature and target descriptions are as follows:

| Feature Variable | Feature Name | Details |
|------------------|--------------|---------|
|X1 | Perimeter_ratio | The ratio between the top and base perimeters. |
|X2 | c4_base | The parameter controlling the size and shape of the base 4-lobe feature. |
|X3 | c8_base | The parameter controlling the size and shape of the base 8-lobe feature. |
|X4 | c4_top | The parameter controlling the size and shape of the top 4-lobe feature. |
|X5 | c8_top | The parameter controlling the size and shape of the top 8-lobe feature. |
|X6 | twist_linear | The rotation (rad) of the top. This creates a linear twist between the base and top. |
|X7 | twist_amplitude | The amplitude (rad) of the oscillating twist between the base and top. |
|X8 | twist_cycles | The number of cycles of the oscillating twist between the base and top. |
|X9 | height | The height (mm). |
|X10 | mass | The mass (g). |
|X11 | thickness | The wall thickness (mm). |
|y | target variable | Energy absorption by the 3D printed structure. |


A training subset of 3000 points is chosen to represent the experiments performed during commissioning over a course of 4-5 months with 20-30 samples a day. 
The remainder of the data is isolated to prevent data leakage. 


In [ ]:
from sklearn.model_selection import train_test_split

df = df.rename(columns={"Unnamed: 0": "Pool_index"})
features = ["x1","x2","x3","x4","x5","x6","x7","x8","x9","x10","x11"]

X = df[features].values
y = df["y"].values

# ── Scalers fitted on full data so pool scoring stays consistent ───────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

# ── Stratified subsample for GP fitting (keeps y distribution intact) ──
N_TRAIN = 3000   # increase to 3000-4000 if accuracy is insufficient

np.random.seed(1003)

# Shuffle the full dataset first, then stratify
shuffle_idx = np.random.permutation(len(y_scaled))
X_shuffled  = X_scaled[shuffle_idx]
y_shuffled  = y_scaled[shuffle_idx]

# Stratified subsample on shuffled data
bins      = np.percentile(y_shuffled, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_shuffled >= bins[i]) & (y_shuffled < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)

train_idx = np.array(train_idx)

X_train = X_shuffled[train_idx]
y_train = y_shuffled[train_idx]

training_shuffled = df.iloc[shuffle_idx].reset_index(drop=True)

print(f"GP training subset : {X_train.shape[0]} points")
print(f"Pool_index range in training subset: {training_shuffled.iloc[train_idx]['Pool_index'].min()} – {training_shuffled.iloc[train_idx]['Pool_index'].max()}")

print(f"GP training subset : {X_train.shape[0]} points (used for fitting)")
print(f"X range : [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")
print(f"y range : [{y_scaled.min():.4f}, {y_scaled.max():.4f}]")

MinMax scaling was chosen to capture all eleven features are physically bounded by the experimental design space i.e. wall thickness can't be negative, twist amplitude can't be arbitrarily large. 
MinMax normalization to [0,1] preserves the bounded structure of the design space without distorting it, which matters because the GP kernel operates directly in that scaled space.

In [ ]:
#  HISTOGRAMS  –  feature distributions


# If X_train is a numpy array, convert it first
if not isinstance(X_train, pd.DataFrame):
    X_train_df = pd.DataFrame(X_train, columns=features)
else:
    X_train_df = X_train

# If y_train is a numpy array, convert it first
if not isinstance(y_train, pd.Series):
    y_train_df = pd.Series(y_train, name='y')
else:
    y_train_df = y_train

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(X_train_df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Scaled Value')
    axes[i].set_ylabel('Count')
    axes[i].grid(axis='y', linestyle='--', alpha=0.5)

# Last subplot: target y
axes[11].hist(y_train, bins=40, color='darkorange', edgecolor='white', alpha=0.85)
axes[11].set_title('y  (target)', fontsize=12, fontweight='bold')
axes[11].set_xlabel('y')
axes[11].set_ylabel('Count')
axes[11].grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('Feature & Target Distributions (Training Set, n=3000)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('histograms.png', dpi=150, bbox_inches='tight')
plt.show()


From the histograms of the training data, we can see that related to its scaled value: 
 - x1, x2, x6, x7, x8 are all right-skewed
    - these correspond to the perimeter ratio, c4_base, and twist features
    - this are features that tend to smaller values, likely limited by 
- x3 and x9 are bimodal
    - these represent the c8_base and height
    - varies between low and high values
- x4, x5, x10, x11 are normally distributed
    - these represent c4_top, c8_top, mass and thickness
    - more flexible variables with respect to 3D printing

The target variable appears to be a combination of normal and bimodal at the tails. At initial inspection the two bimodel features (one or both of: c8_base and height) are expected to have a substantial influence on the target variable. However, the normal distribution also indicates the impact of the rest of the features. 

In [ ]:
#  CORRELATION HEATMAP
corr_df = X_train_df.copy()
corr_df['y'] = y_train_df
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # show lower triangle only

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    ax=ax
)
ax.set_title('Pearson Correlation Heatmap (Features + Target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


The high influence of features of x3 and x9 show up in the heatmap as well.

In [ ]:
# PCA  –  dimensionality reduction

pca = PCA(random_state=403)
pca.fit(X_train_df)
X_pca = pca.transform(X_train_df)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# (a) Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_var)+1), explained_var * 100,
            color='steelblue', edgecolor='white')
axes[0].plot(range(1, len(explained_var)+1), explained_var * 100,
             'o-', color='navy', markersize=5)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var * 100,
             's-', color='darkorange')
axes[1].axhline(90, color='red', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.5)

plt.suptitle('PCA Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPCs needed for 90% variance: "
      f"{np.argmax(cumulative_var >= 0.90) + 1}")

# (b) PC1 vs PC2 scatter — coloured by target y
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                c=y_train, cmap='viridis', alpha=0.6, s=15, edgecolors='none')
plt.colorbar(sc, ax=ax, label='y (target)')
ax.set_xlabel(f'PC1  ({explained_var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2  ({explained_var[1]*100:.1f}%)')
ax.set_title('PCA: PC1 vs PC2 (coloured by target y)', fontsize=13, fontweight='bold')
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()





From the PCA, it is clear that there is a cluster with a high target variable. This cluster should be prioritized for scientific discovery. 

In [ ]:
from scipy.stats import f_oneway

# ─────────────────────────────────────────────
# 4. ANOVA  –  each feature vs. target y
# ─────────────────────────────────────────────
# Strategy: bin y into 5 equal-frequency groups,
# then test whether feature means differ across bins.

"""n_bins = 5
y_binned = pd.qcut(y_train_df, q=n_bins, labels=[f'Q{i+1}' for i in range(n_bins)])

anova_results = []
for col in features:
    groups = [X_train_df.loc[y_binned == label, col].values
              for label in y_binned.categories]
    f_stat, p_val = f_oneway(*groups)
    anova_results.append({'Feature': col, 'F-statistic': f_stat, 'p-value': p_val})"""

n_bins = 5
y_binned = pd.Series(
    pd.qcut(y_train, q=n_bins, labels=[f'Q{i+1}' for i in range(n_bins)])
)  # wrap in Series so .cat accessor and boolean indexing work

anova_results = []
for col in features:
    groups = [X_train_df.loc[y_binned == label, col].values
              for label in y_binned.cat.categories]  # Series needs .cat accessor
    f_stat, p_val = f_oneway(*groups)
    anova_results.append({'Feature': col, 'F-statistic': f_stat, 'p-value': p_val})

anova_df = pd.DataFrame(anova_results).sort_values('F-statistic', ascending=False)
anova_df['Significant (α=0.05)'] = anova_df['p-value'] < 0.05
print("\nANOVA Results (feature vs y quintile bins):")
print(anova_df.to_string(index=False))

# (a) Bar chart of F-statistics
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if sig else 'lightcoral'
          for sig in anova_df['Significant (α=0.05)']]
bars = ax.barh(anova_df['Feature'], anova_df['F-statistic'],
               color=colors, edgecolor='white')
ax.set_xlabel('F-statistic')
ax.set_title('One-Way ANOVA: F-statistics per Feature\n'
             '(blue = significant p<0.05, red = not significant)',
             fontsize=13, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('anova_fstats.png', dpi=150, bbox_inches='tight')
plt.show()




The ANOVA analysis also confirms that x3 and x9 are likely to have the most significant affect on target variable.

With an understanding of the training data and its features, the project cycle is as follows:

1. (Re)Train surrogate model
2. (Receive new proposals and feedback)
3. (Update) Acquisition policy
4. Collect data and submit top points

Brackets indicate steps that occur after the first iteration. 

The key design challenge is that each iteration gives new information about model calibration, community satisfaction, and which regions of feature space remain unexplored. 

The acquisition policy must be updated accordingly.

## Picking the acquisiton policy


### Iteration 1:

Multi-Objective Optimization - Acquisition Scoring

$$S(x) = \alpha \cdot \text{UCB}_{\text{norm}}(x) + \beta \cdot d_{\text{norm}}(x), \quad \alpha = 0.6,\ \beta = 0.4$$



Objective 1: Scientific Discovery

$$\text{UCB}(x) = \mu(x) + \kappa \cdot \sigma(x), \quad \kappa = 2.0$$

Objective 2: Community Engagement

$$d(x) = \frac{1}{k} \sum_{i=1}^{k} \left\| x - x_i \right\|_2, \quad k = 20$$

Weight Rationale

In iteration 1, the acquisition function was a two-objective weighted combination:
The first objective is scientific discovery via Upper Confidence Bound with κ = 2.0, balancing exploitation of high predicted mean with exploration of high uncertainty. κ = 2.0 was chosen to emphasize exploration in an early iteration where the surrogate has limited data. 

The second objective is community engagement via mean distance to the k = 20 nearest neighbours in PCA-reduced feature space.. This is a diversity metric that rewards points far from their neighbours, ensuring geographic spread across the design space.

α=0.6 prioritises model-driven discovery as the primary objective.

β=0.4 ensures geographic spread across feature space, satisfying the community engagement requirement for diverse, interpretable experiment selection.

In [ ]:
def objective_scientific_discovery(X_scaled, gp, kappa=2.0):
    """
    Objective 1: Scientific Discovery
    UCB(x) = mu(x) + kappa * sigma(x)
    
    Parameters
    ----------
    X_scaled : np.ndarray, shape (n_samples, n_features) — scaled feature matrix
    gp       : fitted GaussianProcessRegressor
    kappa    : float — exploration-exploitation trade-off (default 2.0)

    Returns
    -------
    ucb      : np.ndarray, shape (n_samples,) — raw UCB scores
    mu       : np.ndarray, shape (n_samples,) — posterior mean predictions
    sigma    : np.ndarray, shape (n_samples,) — posterior standard deviations
    """
    mu, sigma = gp.predict(X_scaled, return_std=True)
    ucb = mu + kappa * sigma
    return ucb, mu, sigma


def objective_community_engagement(X_scaled, n_components=5, k=20, random_state=1003):
    """
    Objective 2: Community Engagement
    d(x) = (1/k) * sum_{i=1}^{k} ||x - x_i||_2
    
    Mean distance to k nearest neighbours in PCA-reduced feature space.
    Large d(x) indicates a sparse, underexplored region of the geometry space.

    Parameters
    ----------
    X_scaled    : np.ndarray, shape (n_samples, n_features) — scaled feature matrix
    n_components: int   — number of PCA components (default 5)
    k           : int   — number of nearest neighbours (default 20)
    random_state: int   — random seed

    Returns
    -------
    d           : np.ndarray, shape (n_samples,) — mean k-NN distances
    X_pca       : np.ndarray, shape (n_samples, n_components) — PCA-reduced features
    """
    pca = PCA(n_components=n_components, random_state=random_state)
    X_pca = pca.fit_transform(X_scaled)

    nn = NearestNeighbors(n_neighbors=k + 1)  # +1 to exclude self
    nn.fit(X_pca)
    distances, _ = nn.kneighbors(X_pca)
    d = distances[:, 1:].mean(axis=1)  # exclude self (distance = 0)
    return d, X_pca


def combined_acquisition_score(X_scaled, gp, alpha=0.6, beta=0.4, kappa=2.0, k=20, n_components=5):
    """
    Combined acquisition score:
    S(x) = alpha * UCB_norm(x) + beta * d_norm(x)

    Parameters
    ----------
    X_scaled    : np.ndarray — scaled feature matrix
    gp          : fitted GaussianProcessRegressor
    alpha       : float — weight for scientific discovery (default 0.6)
    beta        : float — weight for community engagement (default 0.4)
    kappa       : float — UCB exploration parameter (default 2.0)
    k           : int   — number of nearest neighbours for d(x) (default 20)
    n_components: int   — PCA components for d(x) (default 5)

    Returns
    -------
    score       : np.ndarray — combined acquisition scores
    mu          : np.ndarray — GP posterior means
    sigma       : np.ndarray — GP posterior standard deviations
    """
    assert np.isclose(alpha + beta, 1.0), "alpha + beta must sum to 1"

    ucb, mu, sigma = objective_scientific_discovery(X_scaled, gp, kappa=kappa)
    d, _ = objective_community_engagement(X_scaled, n_components=n_components, k=k)

    ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
    d_norm   = (d   - d.min())   / (d.max()   - d.min())

    score = alpha * ucb_norm + beta * d_norm
    return score, mu, sigma

In [ ]:
class SurrogateGP:
    """
    Gaussian Process surrogate model for multi-objective Bayesian optimisation.

    Parameters
    ----------
    n_train      : int   — number of points to subsample for fitting (default 2000)
    n_strata     : int   — number of quantile strata for stratified sampling (default 20)
    kappa        : float — UCB exploration-exploitation parameter (default 2.0)
    n_components : int   — PCA components for community engagement objective (default 5)
    k_neighbors  : int   — k for mean k-NN distance d(x) (default 20)
    alpha        : float — weight for scientific discovery in S(x) (default 0.6)
    beta         : float — weight for community engagement in S(x) (default 0.4)
    random_state : int   — random seed (default 1003)
    """

    def __init__(
        self,
        n_train=2000,
        n_strata=20,
        kappa=2.0,
        n_components=5,
        k_neighbors=20,
        alpha=0.6,
        beta=0.4,
        random_state=1003,
    ):
        assert np.isclose(alpha + beta, 1.0), "alpha + beta must sum to 1"

        self.n_train      = n_train
        self.n_strata     = n_strata
        self.kappa        = kappa
        self.n_components = n_components
        self.k_neighbors  = k_neighbors
        self.alpha        = alpha
        self.beta         = beta
        self.random_state = random_state

        self.scaler_ = StandardScaler()
        self.gp_     = GaussianProcessRegressor(
            kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
            n_restarts_optimizer=5,
            normalize_y=True,
            random_state=random_state,
        )
        self.is_fitted_ = False

    def _stratified_subsample(self, y):
        np.random.seed(self.random_state)
        bins = np.percentile(y, np.linspace(0, 100, self.n_strata + 1))
        indices = []
        per_stratum = max(1, self.n_train // self.n_strata)
        for i in range(self.n_strata):
            mask = (y >= bins[i]) & (y < bins[i + 1])
            candidates = np.where(mask)[0]
            chosen = np.random.choice(
                candidates, min(per_stratum, len(candidates)), replace=False
            )
            indices.extend(chosen)
        return np.array(indices)


    def fit(self, X, y):
        self.gp_.fit(X, y)
        self.X_scaled_ = X
        self.y_        = y
        self.is_fitted_ = True

        print(f"GP fitted on {len(y)} / {len(y)} points.")
        print(f"Optimised kernel: {self.gp_.kernel_}")
        return self

    def score(self):
        if not self.is_fitted_:
            raise RuntimeError("Call .fit(X, y) before scoring.")

        ucb, mu, sigma = objective_scientific_discovery(self.X_scaled_, self.gp_, kappa=self.kappa)
        d, _           = objective_community_engagement(self.X_scaled_, n_components=self.n_components, k=self.k_neighbors)
        S, mu, sigma   = combined_acquisition_score(self.X_scaled_, self.gp_,
                                                    alpha=self.alpha, beta=self.beta,
                                                    kappa=self.kappa, k=self.k_neighbors,
                                                    n_components=self.n_components)
        return S, mu, sigma

    def select_top_n(self, n=26):
        S, mu, sigma = self.score()
        top_idx = np.argsort(S)[::-1][:n]
        return top_idx, S, mu, sigma

In [ ]:
model = SurrogateGP(random_state=1003)
model.fit(X_train, y_train)

model.X_scaled_ = X_shuffled
model.y_        = y_shuffled

top_idx, S, mu, sigma = model.select_top_n(n=26)

In [ ]:
top_idx

In [ ]:
def make_explanation(row):
    feat_cols = [f"x{i}" for i in range(1, 12)]
    top_feat  = row[feat_cols].abs().idxmax()
    return (
        f"Selected for high combined acquisition score (S={row['score']:.3f}) "
        f"driven by predicted energy absorption mu={row['mu']:.3f} and model uncertainty sigma={row['sigma']:.3f} "
        f"(scientific discovery), and high feature-space diversity (community engagement). "
        f"Most dominant feature: {top_feat}={row[top_feat]:.3f}."
    )

# ── Build submission dataframe ─────────────────────────────────────────
submission = df.iloc[top_idx].copy().reset_index(drop=True)
submission = submission.rename(columns={name: f"x{i+1}" for i, name in enumerate(features)})

submission["mu"]    = mu[top_idx]
submission["sigma"] = sigma[top_idx]
submission["score"] = S[top_idx]
submission["explanation"] = submission.apply(make_explanation, axis=1)
submission["sis_id"] = "ansarir5"

# ── Reorder to match submission spec ──────────────────────────────────
feat_cols  = [f"x{i}" for i in range(1, 12)]
submission = submission[["sis_id", "Pool_index"] + feat_cols + ["y", "score", "explanation"]]

submission = submission.sort_values("score", ascending=False).reset_index(drop=True)

print(submission[["sis_id", "Pool_index", "y", "score"]].to_string(index=False))

In [ ]:
submission

In [ ]:
submission.to_csv("iteration1_submission2.csv", index=False)
print("Saved.")

In [ ]:
from sklearn.model_selection import train_test_split

df = df.rename(columns={"Unnamed: 0": "Pool_index"})
features = ["x1","x2","x3","x4","x5","x6","x7","x8","x9","x10","x11"]

X = df[features].values
y = df["y"].values

# ── Scalers fitted on full data so pool scoring stays consistent ───────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

# ── Stratified subsample for GP fitting (keeps y distribution intact) ──
N_TRAIN = 3000   # increase to 3000-4000 if accuracy is insufficient

np.random.seed(1003)

# Shuffle the full dataset first, then stratify
shuffle_idx = np.random.permutation(len(y_scaled))
X_shuffled  = X_scaled[shuffle_idx]
y_shuffled  = y_scaled[shuffle_idx]

# Stratified subsample on shuffled data
bins      = np.percentile(y_shuffled, np.linspace(0, 100, 21))
train_idx = []
per_bin   = max(1, N_TRAIN // 20)
for i in range(len(bins) - 1):
    mask       = (y_shuffled >= bins[i]) & (y_shuffled < bins[i + 1])
    candidates = np.where(mask)[0]
    chosen     = np.random.choice(candidates, min(per_bin, len(candidates)), replace=False)
    train_idx.extend(chosen)

train_idx = np.array(train_idx)

X_train = X_shuffled[train_idx]
y_train = y_shuffled[train_idx]

# Keep full shuffled pool for scoring, with matching Pool_index
training_shuffled = df.iloc[shuffle_idx].reset_index(drop=True)

print(f"GP training subset : {X_train.shape[0]} points")
print(f"Pool_index range in training subset: {training_shuffled.iloc[train_idx]['Pool_index'].min()} – {training_shuffled.iloc[train_idx]['Pool_index'].max()}")

print(f"X range : [{X_scaled.min():.4f}, {X_scaled.max():.4f}]")
print(f"y range : [{y_scaled.min():.4f}, {y_scaled.max():.4f}]")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.gridspec as gridspec


# ── Train/test split on augmented subsampled data ──────────────────────
X_tr_eval, X_te_eval, y_tr_eval, y_te_eval = train_test_split(
    X_train, y_train, test_size=0.2, random_state=1003
)

gp_eval = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval.fit(X_tr_eval, y_tr_eval)
mu_test, sigma_test = gp_eval.predict(X_te_eval, return_std=True)

rmse     = np.sqrt(mean_squared_error(y_te_eval, mu_test))
r2       = 1 - np.sum((y_te_eval - mu_test)**2) / np.sum((y_te_eval - y_te_eval.mean())**2)
mean_std = sigma_test.mean()

print(f"Test RMSE : {rmse:.4f}")
print(f"Test R²   : {r2:.4f}")
print(f"Mean σ    : {mean_std:.4f}")

# ── Full augmented pool predictions from fitted surrogate ──────────────
S, mu, sigma = model.score()
d, X_pca     = objective_community_engagement(
    model.X_scaled_, n_components=model.n_components, k=model.k_neighbors
)
ucb      = mu + model.kappa * sigma
ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm   = (d   - d.min())   / (d.max()   - d.min())

# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.suptitle(
    "Iteration 1 — Surrogate Performance, Uncertainty & Acquisition Policy",
    fontsize=14, fontweight="bold", y=0.99
)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit = fig.add_subplot(gs[0, :2])
ax_res = fig.add_subplot(gs[0, 2:])
ax_ucb = fig.add_subplot(gs[1, 0])
ax_std = fig.add_subplot(gs[1, 1])
ax_div = fig.add_subplot(gs[1, 2])
ax_s   = fig.add_subplot(gs[1, 3])

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_te_eval, mu_test, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_te_eval.min(), mu_test.min()), max(y_te_eval.max(), mu_test.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse:.4f}  R²={r2:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test.min(), sigma_test.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals = y_te_eval - mu_test
ax_res.scatter(mu_test, residuals, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test.min(), mu_test.max(), 100),
    -2 * sigma_test.mean(), 2 * sigma_test.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper: distribution panel ───────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 26 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6: Score component distributions ────────────────────────
dist_panel(ax_ucb, ucb_norm,  ucb_norm[top_idx],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution",      "#3498db")
dist_panel(ax_std, sigma,     sigma[top_idx],
           "σ(x)",            "Model Uncertainty\nσ distribution",            "#9b59b6")
dist_panel(ax_div, d_norm,    d_norm[top_idx],
           "$d_{norm}$(x)",   "Community Engagement\nDiversity distribution", "#2ecc71")
dist_panel(ax_s,   S,         S[top_idx],
           "S(x)",            "Combined Acquisition\nScore S(x)",             "#e67e22")

#plt.savefig("/mnt/user-data/outputs/iteration2_performance.png", dpi=150, bbox_inches="tight")
#plt.show()
#print("Saved.")

Before trusting the acquisition policy to guide experiment selection, the surrogate model was evaluated with a held out 20% of the stratified training set as a blind test set. Evaluating on training data would be misleading since GPs interpolate exactly through their own training points.

An R² of 0.13 is a weak fit. But the critical observation is that mean σ ≈ RMSE, meaning the GP is well-calibrated. It knows what it doesn't know. For UCB-based acquisition, a well-calibrated uncertain model is more useful than an overconfident poor one, because the exploration term is meaningful. The selected points came from the upper tail of the UCB and uncertainty distributions, and were reasonably distributed across PCA feature space.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Gather scores and objectives for all points ────────────────────────
S, mu, sigma = model.score()
d, X_pca     = d, X_pca = objective_community_engagement(model.X_scaled_, n_components=model.n_components, k=model.k_neighbors)
#model.objective_community_engagement()

ucb          = mu + model.kappa * sigma
ucb_norm     = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm       = (d   - d.min())   / (d.max()   - d.min())

# ── Pareto frontier (maximise both objectives) ─────────────────────────
# A point is non-dominated if no other point is better in both objectives
objectives   = np.stack([ucb_norm, d_norm], axis=1)   

def is_non_dominated(obj):
    """Returns boolean mask of non-dominated points (maximisation)."""
    n        = len(obj)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(obj[j] >= obj[i]) and np.any(obj[j] > obj[i]):
                dominated[i] = True
                break
    return ~dominated

is_pareto    = is_non_dominated(objectives[np.argsort(S)[::-1][:500]])  # check on top 500 for speed
pareto_idx   = np.argsort(S)[::-1][:500][is_pareto]
top26_mask   = np.isin(np.arange(len(S)), top_idx)

# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
fig.suptitle("Iteration 1 — Acquisition Policy: Pareto Frontier & Objective Space",
             fontsize=14, fontweight="bold", y=0.99)

gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
ax_s = fig.add_subplot(gs[0, :])   # full-width: score vs index
ax_o = fig.add_subplot(gs[1, 0])   # objective space: UCB vs diversity
ax_p = fig.add_subplot(gs[1, 1])   # PCA coloured by S

# ─── Panel 1: Acquisition score across all pool points ────────────────
ax_s.scatter(np.arange(len(S)), S,
             c=S, cmap="viridis", s=2, alpha=0.3, rasterized=True)
ax_s.scatter(top_idx, S[top_idx],
             c="red", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, label="Top 26 selected")
ax_s.scatter(pareto_idx, S[pareto_idx],
             c="gold", s=40, zorder=4, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated (top 500)")
ax_s.set_xlabel("Pool index")
ax_s.set_ylabel("Acquisition score S(x)")
ax_s.set_title("Acquisition Score")
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.2)

# ─── Panel 2: Objective space with Pareto frontier ────────────────────
# All points (faded)
ax_o.scatter(ucb_norm, d_norm,
             c="steelblue", s=3, alpha=0.15, rasterized=True, label="All points")

# Pareto-non-dominated (top 500 subset)
ax_o.scatter(ucb_norm[pareto_idx], d_norm[pareto_idx],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, marker="*", label="Pareto-non-dominated")

# Top 26 selected
ax_o.scatter(ucb_norm[top_idx], d_norm[top_idx],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 26 selected")

# Pareto staircase
pareto_pts = np.stack([ucb_norm[pareto_idx], d_norm[pareto_idx]], axis=1)
sorted_p   = pareto_pts[np.argsort(pareto_pts[:, 0])]
ax_o.step(sorted_p[:, 0], sorted_p[:, 1], where="post",
          color="gold", lw=2, ls="--", alpha=0.85, zorder=4, label="Pareto front")

ax_o.set_xlabel("UCB$_{norm}$(x) — Scientific Discovery")
ax_o.set_ylabel("$d_{norm}$(x) — Community Engagement")
ax_o.set_title("Objective Space with Pareto Frontier")
ax_o.legend(fontsize=9)
ax_o.grid(True, alpha=0.2)

# ─── Panel 3: PCA space coloured by acquisition score ─────────────────
sc = ax_p.scatter(X_pca[:, 0], X_pca[:, 1],
                  c=S, cmap="viridis", s=3, alpha=0.3, rasterized=True)
ax_p.scatter(X_pca[pareto_idx, 0], X_pca[pareto_idx, 1],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated")
ax_p.scatter(X_pca[top_idx, 0], X_pca[top_idx, 1],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 26 selected")
ax_p.set_xlabel("PC1")
ax_p.set_ylabel("PC2")
ax_p.set_title("PCA Feature Space\n(colour = acquisition score)")
ax_p.legend(fontsize=9)
ax_p.grid(True, alpha=0.2)
plt.colorbar(sc, ax=ax_p, label="S(x)")



#plt.savefig("/mnt/user-data/outputs/iteration1_pareto.png", dpi=150, bbox_inches="tight")
#plt.show()
#print("Saved.")

Looking at the objective space and PCA feature space plots for iteration 1, the selected points cluster near the Pareto frontier and are fairly well distributed across feature space. 

The model is finding points that are both high-performing and diverse. The Pareto plot also shows we're selecting genuinely non-dominated points, which validates that the two objectives are working together rather than against each other for the first iteration.

For the next selection, the acquisition policy needs to consider a distribution of the selected proposals for community engagement. 

### Iteration 2

Now that we have received 13 proposals with 20 suggested points each, we have to select the top 20 points for evaluation. 

Now that there are actual proposals to choose from, the acquistion function was modified to include the community norm and coverage norm:

$$S(x) = \alpha \cdot \text{UCB}_{\text{norm}}(x) + \beta \cdot d_{\text{norm}}(x) + \gamma \cdot \text{cs}_{\text{norm}}(x) + \delta \cdot \text{cov}_{\text{norm}}(x)$$
$$ \alpha = 0.4, \quad \beta = 0.3, \quad \gamma = 0.2, \quad \delta = 0.1, \quad \sum = 1.0 $$



$$ \text{cs}_{\text{norm}}(x) = \frac{\text{cs}(x) - \min(\text{cs})}{\max(\text{cs}) - \min(\text{cs}) + \epsilon}, \quad \epsilon = 10^{-8}\
{}$$

$$\text{cov}(x) = \frac{1}{\text{freq}(\text{scientist\_id})}$$


I reduced α from 0.6 to 0.4, to explicitly incorporate the community scientist score cs_norm, which encodes what scientists actually proposed and a scientist coverage term cov_norm, which scores points by the inverse frequency of their scientist ID. The coverage term ensures no single scientist's proposals dominate across rounds.



In [ ]:
def acquisition_score_iter2(
    model,
    proposals_full,
    pool_idx_shuf,
    alpha=0.4,    # scientific discovery (down from 0.6)
    beta=0.3,     # feature-space diversity
    gamma=0.2,    # community scientist score
    delta=0.1,    # scientist coverage
    kappa=2.0,
    n_components=5,
    k=20,
):
    """
    Updated acquisition score for iteration 2.

    S(x) = alpha * UCB_norm(x)
         + beta  * d_norm(x)
         + gamma * community_score_norm(x)
         + delta * scientist_coverage_norm(x)

    Only scores points that appear in proposals_full (the 200-point proposal pool).
    
    Parameters
    ----------
    model             : fitted SurrogateGP (scored over full augmented pool)
    proposals_full    : pd.DataFrame — pooled proposals with Pool_index, 
                        community_score, scientist_id, and x1–x11
    pool_idx_shuf     : np.ndarray — Pool_index values aligned to model.X_scaled_
    alpha             : weight for UCB (scientific discovery)
    beta              : weight for feature-space diversity
    gamma             : weight for community scientist score
    delta             : weight for scientist coverage
    """
    assert np.isclose(alpha + beta + gamma + delta, 1.0), "weights must sum to 1"

    # ── Locate proposal rows in the augmented pool ─────────────────────
    proposal_pool_indices = proposals_full["Pool_index"].values
    mask = np.isin(pool_idx_shuf, proposal_pool_indices)
    proposal_rows = np.where(mask)[0]   # positional indices into model.X_scaled_

    # Align proposals_full to the order they appear in model.X_scaled_
    idx_map = {pid: pos for pos, pid in zip(proposal_rows,
               pool_idx_shuf[proposal_rows])}
    aligned_pos = np.array([idx_map[pid] for pid in proposal_pool_indices
                            if pid in idx_map])
    aligned_proposals = proposals_full[
        proposals_full["Pool_index"].isin(pool_idx_shuf[aligned_pos])
    ].copy().reset_index(drop=True)

    X_prop = model.X_scaled_[aligned_pos]   # shape (n_proposals, 11)

    # ── Objective 1: Scientific Discovery — UCB ────────────────────────
    ucb, mu, sigma = objective_scientific_discovery(X_prop, model.gp_, kappa=kappa)
    ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min() + 1e-8)

    # ── Objective 2: Feature-space diversity — k-NN in PCA space ──────
    d, _ = objective_community_engagement(
        X_prop, n_components=n_components, k=min(k, len(X_prop) - 1)
    )
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)

    # ── Objective 3: Community scientist score ─────────────────────────
    """cs = aligned_proposals["community_score"].values.astype(float)
    cs_norm = (cs - cs.min()) / (cs.max() - cs.min() + 1e-8)
"""
    # ── Objective 3: Community scientist score ─────────────────────────────
    cs = aligned_proposals["community_score"].values.astype(float)

    # Replace inf with nan, then fill with column max of finite values
    cs = np.where(np.isinf(cs), np.nan, cs)
    finite_max = np.nanmax(cs)
    finite_min = np.nanmin(cs)
    cs = np.where(np.isnan(cs), finite_max, cs)   # treat inf as highest finite score

    cs_norm = (cs - finite_min) / (finite_max - finite_min + 1e-8)

    # ── Objective 4: Scientist coverage ───────────────────────────────
    # Reward points from scientists who are underrepresented so far.
    # Coverage score = 1 / (frequency of that scientist_id in pool)
    # so rare scientists get a higher score.
    scientist_ids   = aligned_proposals["scientist_id"].values
    id_counts       = pd.Series(scientist_ids).value_counts()
    coverage        = np.array([1.0 / id_counts[sid] for sid in scientist_ids])
    coverage_norm   = (coverage - coverage.min()) / (coverage.max() - coverage.min() + 1e-8)

    # ── Combined score ─────────────────────────────────────────────────
    S = (alpha * ucb_norm
       + beta  * d_norm
       + gamma * cs_norm
       + delta * coverage_norm)

    return S, mu, sigma, ucb_norm, d_norm, cs_norm, coverage_norm, aligned_proposals, aligned_pos

In [ ]:
# ITERATION 2 — STEP 1: Load proposals and score using model (iter 1)

import os, glob
import pandas as pd
import numpy as np

PROPOSAL_DIR = "Proposals_iter1/"
proposal_files = sorted(glob.glob(os.path.join(PROPOSAL_DIR, "proposals_*.csv")))
print(f"Found {len(proposal_files)} proposal files:")
for f in proposal_files:
    print(f"  {os.path.basename(f)}")

COL_MAP = {
    "Index"       : "Pool_index",
    "Prediction"  : "community_pred",
    "Uncertainty" : "community_uncert",
    "Score"       : "community_score",
    "pool_id"     : "Pool_index",
    "prediction"  : "community_pred",
    "std"         : "community_uncert",
    "score"       : "community_score",
}

proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp = tmp.rename(columns=COL_MAP)
    tmp["scientist_id"] = scientist_id
    tmp = tmp[["Pool_index", "community_pred", "community_uncert",
               "community_score", "scientist_id"]]
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"\nPooled : {proposals_pooled.shape[0]} rows from "
      f"{proposals_pooled['scientist_id'].nunique()} scientists")

# Join features and true y
proposals_full_iter2 = proposals_pooled.merge(
    df[["Pool_index"] + features + ["y"]], on="Pool_index", how="left"
)

# Clean inf/nan
proposals_full_iter2["community_score"] = proposals_full_iter2["community_score"].replace(
    [np.inf, -np.inf], np.nan
)
proposals_full_iter2["community_score"] = proposals_full_iter2.groupby("scientist_id")[
    "community_score"
].transform(lambda x: x.fillna(x.median()))
global_median_iter2 = proposals_full_iter2["community_score"].median()
proposals_full_iter2["community_score"] = proposals_full_iter2["community_score"].fillna(
    global_median_iter2
)

# Deduplicate
proposals_deduped_iter2 = (
    proposals_full_iter2
    .groupby("Pool_index")
    .agg(
        community_score  = ("community_score", "mean"),
        scientist_id     = ("scientist_id",     lambda x: ",".join(sorted(set(x)))),
        community_pred   = ("community_pred",   "mean"),
        community_uncert = ("community_uncert", "mean"),
        **{f: (f, "first") for f in features},
        y                = ("y", "first"),
    )
    .reset_index()
)

print(f"Unique proposals after dedup : {len(proposals_deduped_iter2)}")

# Pool_index values aligned to X_shuffled — needed for proposal scoring
pool_idx_shuf = training_shuffled["Pool_index"].values

# Score using model (iter 1 surrogate) — before any retraining
(S_prop2, mu_prop2, sigma_prop2,
 ucb_norm_prop2, d_norm_prop2,
 cs_norm_prop2, cov_norm_prop2,
 aligned_proposals2, aligned_pos2) = acquisition_score_iter2(
    model,                        # iter 1 surrogate
    proposals_deduped_iter2,
    pool_idx_shuf,                # pool_idx_shuf from iter 1 training cell
    alpha=0.9,
    beta=0.05,
    gamma=0.025,
    delta=0.025,
)

print(f"\nScored {len(S_prop2)} unique proposal points")
print(f"Score range: [{S_prop2.min():.4f}, {S_prop2.max():.4f}]")

In [ ]:
# ITERATION 2 — STEP 2: Select top 20

top20_local2 = np.argsort(S_prop2)[::-1][:20]
top20_pos2   = aligned_pos2[top20_local2]
top20_pool2  = pool_idx_shuf[top20_pos2]

results_iter2 = aligned_proposals2.iloc[top20_local2].copy().reset_index(drop=True)
results_iter2["y_pred"]        = mu_prop2[top20_local2]
results_iter2["y_uncert"]      = sigma_prop2[top20_local2]
results_iter2["ucb_norm"]      = ucb_norm_prop2[top20_local2]
results_iter2["d_norm"]        = d_norm_prop2[top20_local2]
results_iter2["cs_norm"]       = cs_norm_prop2[top20_local2]
results_iter2["coverage_norm"] = cov_norm_prop2[top20_local2]
results_iter2["score"]         = S_prop2[top20_local2]
top20_pos_plot2                = aligned_pos2[top20_local2]

print(f"Top 20 Pool_index values: {top20_pool2}")
print(results_iter2[["Pool_index", "scientist_id",
                      "community_score", "score"]].to_string(index=False))

In [ ]:
# ITERATION 2 — STEP 3: Add selected 20 to training set

selected_iter2     = proposals_deduped_iter2[
    proposals_deduped_iter2["Pool_index"].isin(top20_pool2)
]
train_pool_indices2 = set(pool_idx_shuf[train_idx])
new_selected_mask2  = ~selected_iter2["Pool_index"].isin(train_pool_indices2)
new_selected2       = selected_iter2[new_selected_mask2]

X_new2 = scaler_X.transform(new_selected2[features].values)
y_new2 = scaler_y.transform(new_selected2["y"].values.reshape(-1, 1)).ravel()

X_train_aug2 = np.vstack([X_shuffled[train_idx], X_new2])
y_train_aug2 = np.concatenate([y_shuffled[train_idx], y_new2])

print(f"Iter 1 training set  : {X_shuffled[train_idx].shape[0]} points")
print(f"New selected points  : {X_new2.shape[0]} points added")
print(f"Iter 2 training set  : {X_train_aug2.shape[0]} points")

This is further emphasized with the lower overall score of the selected points and fewer points that sit on the pareto frontier. 

The selection throughout the PCA feature space remains well spread.

In [ ]:
# ITERATION 2 — STEP 4: Retrain surrogate → model_iter2

# Augment scoring pool with new selected points
X_pool_aug2    = np.vstack([X_shuffled, X_new2])
y_pool_aug2    = np.concatenate([y_shuffled, y_new2])
pool_idx_shuf2 = np.concatenate([
    training_shuffled["Pool_index"].values,
    new_selected2["Pool_index"].values
])

np.random.seed(1003)
shuffle_idx2   = np.random.permutation(len(y_pool_aug2))
X_aug_shuf2    = X_pool_aug2[shuffle_idx2]
y_aug_shuf2    = y_pool_aug2[shuffle_idx2]
pool_idx_shuf2 = pool_idx_shuf2[shuffle_idx2]

model_iter2 = SurrogateGP(random_state=1003)
model_iter2.fit(X_train_aug2, y_train_aug2)
model_iter2.X_scaled_ = X_aug_shuf2
model_iter2.y_        = y_aug_shuf2

print(f"model_iter2 fitted on {X_train_aug2.shape[0]} points")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ── Train/test split on augmented training data ────────────────────────
X_tr_eval, X_te_eval, y_tr_eval, y_te_eval = train_test_split(
    X_train_aug2, y_train_aug2, test_size=0.2, random_state=1003
)

gp_eval = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval.fit(X_tr_eval, y_tr_eval)
mu_test, sigma_test = gp_eval.predict(X_te_eval, return_std=True)

rmse2     = np.sqrt(mean_squared_error(y_te_eval, mu_test))
r2_2       = 1 - np.sum((y_te_eval - mu_test)**2) / np.sum((y_te_eval - y_te_eval.mean())**2)
mean_std2 = sigma_test.mean()

print(f"Test RMSE : {rmse2:.4f}")
print(f"Test R²   : {r2_2:.4f}")
print(f"Mean σ    : {mean_std2:.4f}")

# ── Full pool scores ───────────────────────────────────────────────────
"""ucb_norm_full = ucb_norm
d_norm_full   = d_norm"""

S, mu, sigma = model_iter2.score()
d, X_pca     = objective_community_engagement(
    model_iter2.X_scaled_, n_components=model_iter2.n_components, k=model_iter2.k_neighbors
)
ucb      = mu + model_iter2.kappa * sigma
ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm   = (d   - d.min())   / (d.max()   - d.min())

ucb_norm_full = ucb_norm
d_norm_full   = d_norm

fig = plt.figure(figsize=(18, 11))
fig.suptitle("Iteration 2 — Surrogate Performance, Uncertainty & Acquisition Policy",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit = fig.add_subplot(gs[0, :2])
ax_res = fig.add_subplot(gs[0, 2:])
ax_ucb = fig.add_subplot(gs[1, 0])
ax_std = fig.add_subplot(gs[1, 1])
ax_div = fig.add_subplot(gs[1, 2])
ax_s   = fig.add_subplot(gs[1, 3])

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_te_eval, mu_test, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_te_eval.min(), mu_test.min()), max(y_te_eval.max(), mu_test.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse:.4f}  R²={r2:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test.min(), sigma_test.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals = y_te_eval - mu_test
ax_res.scatter(mu_test, residuals, c=sigma_test, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test.min(), mu_test.max(), 100),
    -2 * sigma_test.mean(), 2 * sigma_test.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper ───────────────────────────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 20 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6 ───────────────────────────────────────────────────────
dist_panel(ax_ucb, ucb_norm_full, ucb_norm_full[top20_pos_plot2],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution",      "#3498db")
dist_panel(ax_std, sigma,         sigma[top20_pos_plot2],
           "σ(x)",                "Model Uncertainty\nσ distribution",        "#9b59b6")
dist_panel(ax_div, d_norm_full,   d_norm_full[top20_pos_plot2],
           "$d_{norm}$(x)",       "Community Engagement\nDiversity distribution", "#2ecc71")
dist_panel(ax_s,   S,             S[top20_pos_plot2],
           "S(x)",                "Combined Acquisition\nScore S(x)",         "#e67e22")

plt.show()


Surrogate metrics for iteration 2 were improved slightly to iteration 1, RMSE 0.2487, R² 0.1695, Mean σ 0.2437, consistent with the calibrated-but-uncertain pattern from before.

The selection mean shifted on the lower side of the UCB peak and closer to higher peak of model uncertainty. This shows the affect of the new acquisition policy is as intended. 

In [ ]:
S, mu, sigma = model_iter2.score()
d, X_pca     = objective_community_engagement(
    model_iter2.X_scaled_, n_components=model_iter2.n_components, k=model_iter2.k_neighbors
)
ucb      = mu + model_iter2.kappa * sigma
ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min())
d_norm   = (d   - d.min())   / (d.max()   - d.min())

# Pareto on top 500 by score
objectives = np.stack([ucb_norm, d_norm], axis=1)
top500_idx = np.argsort(S)[::-1][:500]
is_pareto  = is_non_dominated(objectives[top500_idx])
pareto_idx = top500_idx[is_pareto]

# top20 positional indices into model_iter2.X_scaled_
top20_pos_plot2 = aligned_pos2[top20_local2]

fig = plt.figure(figsize=(16, 11))
fig.suptitle("Iteration 2 — Acquisition Policy: Pareto Frontier & Objective Space",
             fontsize=14, fontweight="bold", y=0.99)

gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
ax_s = fig.add_subplot(gs[0, :])
ax_o = fig.add_subplot(gs[1, 0])
ax_p = fig.add_subplot(gs[1, 1])

# ─── Panel 1: Acquisition score across augmented pool ─────────────────
ax_s.scatter(np.arange(len(S)), S,
             c=S, cmap="viridis", s=2, alpha=0.3, rasterized=True)
ax_s.scatter(top20_pos_plot2, S[top20_pos_plot2],
             c="red", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_s.scatter(pareto_idx, S[pareto_idx],
             c="gold", s=40, zorder=4, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated (top 500)")
ax_s.set_xlabel("Pool index")
ax_s.set_ylabel("Acquisition score S(x)")
ax_s.set_title("Acquisition Score Across Augmented Pool")
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.2)

# ─── Panel 2: Objective space with Pareto frontier ────────────────────
ax_o.scatter(ucb_norm, d_norm,
             c="steelblue", s=3, alpha=0.15, rasterized=True, label="All points")
ax_o.scatter(ucb_norm[pareto_idx], d_norm[pareto_idx],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, marker="*", label="Pareto-non-dominated")
ax_o.scatter(ucb_norm[top20_pos_plot2], d_norm[top20_pos_plot2],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
pareto_pts = np.stack([ucb_norm[pareto_idx], d_norm[pareto_idx]], axis=1)
sorted_p   = pareto_pts[np.argsort(pareto_pts[:, 0])]
ax_o.step(sorted_p[:, 0], sorted_p[:, 1], where="post",
          color="gold", lw=2, ls="--", alpha=0.85, zorder=4, label="Pareto front")
ax_o.set_xlabel("UCB$_{norm}$(x) — Scientific Discovery")
ax_o.set_ylabel("$d_{norm}$(x) — Community Engagement")
ax_o.set_title("Objective Space with Pareto Frontier")
ax_o.legend(fontsize=9)
ax_o.grid(True, alpha=0.2)

# ─── Panel 3: PCA space ───────────────────────────────────────────────
sc = ax_p.scatter(X_pca[:, 0], X_pca[:, 1],
                  c=S, cmap="viridis", s=3, alpha=0.3, rasterized=True)
ax_p.scatter(X_pca[pareto_idx, 0], X_pca[pareto_idx, 1],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated")
ax_p.scatter(X_pca[top20_pos_plot2, 0], X_pca[top20_pos_plot2, 1],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_p.set_xlabel("PC1")
ax_p.set_ylabel("PC2")
ax_p.set_title("PCA Feature Space\n(colour = acquisition score)")
ax_p.legend(fontsize=9)
ax_p.grid(True, alpha=0.2)
plt.colorbar(sc, ax=ax_p, label="S(x)")

#plt.savefig("/mnt/user-data/outputs/iteration2_pareto.png", dpi=150, bbox_inches="tight")
plt.show()
#print("Saved.")

A good distribution of points in the PCA-space were selected, however no non dominated points were selected. This could be due to the acquisition function but it could also be a carry over affect from the last iteration. The scientists might have adjusted their proposals for diversity and suggested less ambitious points. This gives a smaller pool of points that are pushing the pareto envelope. 

In [ ]:
top20_local2 = np.argsort(S_prop2)[::-1][:20]
top20_pos2   = aligned_pos2[top20_local2]
top20_pool2  = pool_idx_shuf[top20_pos2]

results_iter2 = aligned_proposals2.iloc[top20_local2].copy().reset_index(drop=True)
results_iter2["y_pred"]        = mu_prop2[top20_local2]
results_iter2["y_uncert"]      = sigma_prop2[top20_local2]
results_iter2["ucb_norm"]      = ucb_norm_prop2[top20_local2]
results_iter2["d_norm"]        = d_norm_prop2[top20_local2]
results_iter2["cs_norm"]       = cs_norm_prop2[top20_local2]
results_iter2["coverage_norm"] = cov_norm_prop2[top20_local2]
results_iter2["score"]         = S_prop2[top20_local2]
top20_pos_plot2                 = aligned_pos2[top20_local2]

In [ ]:
def make_explanation_iter2(row):
    feat_cols = [f"x{i}" for i in range(1, 12)]
    top_feat  = row[feat_cols].abs().idxmax()
    n_scientists = len(row["scientist_id"].split(","))
    return (
        f"Selected via multi-objective acquisition (S={row['score']:.3f}) "
        f"balancing scientific discovery (UCB: mu={row['y_pred']:.3f}, sigma={row['y_uncert']:.3f}) "
        f"and community engagement (diversity={row['d_norm']:.3f}, "
        f"community_score={row['community_score']:.3f}, "
        f"proposed by {n_scientists} scientist(s): {row['scientist_id']}). "
        f"Most dominant feature: {top_feat}={row[top_feat]:.3f}."
    )

# ── Build submission dataframe ─────────────────────────────────────────
submission_iter2 = results_iter2.copy()

# Rename feature columns to x1–x11
feat_rename = {features[i]: f"x{i+1}" for i in range(11)}
submission_iter2 = submission_iter2.rename(columns=feat_rename)

# Add model predictions aligned to top20
submission_iter2["y_pred"]   = mu_prop2[top20_local2]
submission_iter2["y_uncert"] = sigma_prop2[top20_local2]
submission_iter2["d_norm"]   = d_norm_prop2[top20_local2]
submission_iter2["sis_id"]   = "ansarir5"

submission_iter2["explanation"] = submission_iter2.apply(make_explanation_iter2, axis=1)

# ── Reorder to match submission spec ──────────────────────────────────
feat_cols = [f"x{i}" for i in range(1, 12)]
submission_iter2 = submission_iter2[
    ["sis_id", "Pool_index"] + feat_cols + ["y", "score", "explanation"]
]

submission_iter2 = submission_iter2.sort_values("score", ascending=False).reset_index(drop=True)

print(submission_iter2[["sis_id", "Pool_index", "y", "score"]].to_string(index=False))

In [ ]:
submission_iter2

In [ ]:
submission_iter2.to_csv("iteration2_submission2.csv", index=False)
print("Saved.")

### Iteration 3

Before we submit points for iteration 3, lets understand how happy the community was with the last set of points.

In [ ]:
# ── Parse happiness scores from text file ──────────────────────────────
happiness_raw2 = open("Proposals_iter2/happiness_results.txt").read()

happiness2 = {}
lines = happiness_raw2.strip().split("\n")
current_proposal = None
for line in lines:
    line = line.strip()
    if line.startswith("Proposal file:"):
        current_proposal = line.split("Proposal file:")[-1].strip().replace(".csv", "")
    elif line.startswith("Happiness with samples selected"):
        score = float(line.split(":")[-1].strip())
        happiness2[current_proposal] = score

print("Parsed happiness scores:")
for k, v in sorted(happiness2.items()):
    print(f"  {k}: {v:.4f}")

In [ ]:
results_iter2_1 = pd.read_csv("iteration2_results_full.csv")
print(results_iter2.columns.tolist())

In [ ]:
#comparing iteration2_submission and happiness_results

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd


# ── Map happiness back to each selected point ──────────────────────────
# Points proposed by multiple scientists get the mean happiness
def mean_happiness(scientist_id_str):
    scientists = scientist_id_str.split(",")
    return np.mean([happiness2[s.strip()] for s in scientists])

results_iter2_1["happiness"] = results_iter2_1["scientist_id"].apply(mean_happiness)


# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Iteration 2 — Top 20 Selected Points vs Community Happiness",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

ax_bar  = fig.add_subplot(gs[0, :2])   # happiness per selected point
ax_sci  = fig.add_subplot(gs[0, 2])    # mean happiness per scientist
ax_sco  = fig.add_subplot(gs[1, 0])    # happiness vs acquisition score
ax_ucb  = fig.add_subplot(gs[1, 1])    # happiness vs UCB
ax_div  = fig.add_subplot(gs[1, 2])    # happiness vs diversity

# ─── Panel 1: Happiness per selected point ────────────────────────────
colors = plt.cm.RdYlGn(
    (results_iter2_1["happiness"] - 1) / 4
)
bars = ax_bar.bar(
    range(len(results_iter2_1)),
    results_iter2_1["happiness"],
    color=colors, edgecolor="black", linewidth=0.6
)
ax_bar.axhline(results_iter2_1["happiness"].mean(), color="black",
               lw=1.5, ls="--",
               label=f"Mean happiness = {results_iter2_1['happiness'].mean():.2f}")
ax_bar.set_xticks(range(len(results_iter2_1)))
ax_bar.set_xticklabels(results_iter2_1["Pool_index"], rotation=45, ha="right", fontsize=8)
ax_bar.set_xlabel("Pool_index")
ax_bar.set_ylabel("Happiness score")
ax_bar.set_title("Happiness Score per Selected Point\n(colour: red=low, green=high)")
ax_bar.set_ylim(0, 5.5)
ax_bar.legend(fontsize=9)
ax_bar.grid(True, alpha=0.2, axis="y")

# ─── Panel 2: Mean happiness per scientist ────────────────────────────
sci_df = pd.DataFrame(happiness2.items(), columns=["scientist_id", "happiness"])
sci_df = sci_df.sort_values("happiness", ascending=True)
bar_colors = plt.cm.RdYlGn((sci_df["happiness"] - 1) / 4)
ax_sci.barh(sci_df["scientist_id"], sci_df["happiness"],
            color=bar_colors, edgecolor="black", linewidth=0.6)
ax_sci.axvline(sci_df["happiness"].mean(), color="black", lw=1.5, ls="--",
               label=f"Mean = {sci_df['happiness'].mean():.2f}")
ax_sci.set_xlabel("Happiness score")
ax_sci.set_title("Happiness per Scientist")
ax_sci.set_xlim(0, 5.5)
ax_sci.legend(fontsize=9)
ax_sci.grid(True, alpha=0.2, axis="x")

# ─── Helper: scatter with correlation ─────────────────────────────────
def scatter_corr(ax, x, y, xlabel, title, color):
    ax.scatter(x, y, c=color, s=60, edgecolors="black", linewidths=0.6, alpha=0.8)
    # correlation
    r = np.corrcoef(x, y)[0, 1]
    m, b = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m * xline + b, "k--", lw=1.5, label=f"r = {r:.2f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Happiness")
    ax.set_title(title)
    ax.set_ylim(0, 5.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

# ─── Panels 3–5: Happiness vs score components ────────────────────────
scatter_corr(ax_sco, results_iter2_1["score"],    results_iter2_1["happiness"],
             "Acquisition score S(x)",  "Happiness vs Acquisition Score", "#e67e22")
scatter_corr(ax_ucb, results_iter2_1["ucb_norm"], results_iter2_1["happiness"],
             "UCB$_{norm}$(x)",         "Happiness vs Scientific Discovery", "#3498db")
scatter_corr(ax_div, results_iter2_1["d_norm"],   results_iter2_1["happiness"],
             "$d_{norm}$(x)",           "Happiness vs Diversity", "#2ecc71")

plt.show()

- We can see that mean happiness was 2.57 meaning overall the community was not super happy nor terribly displeased. 

- Overall proposals/scientists 7 and 8 were the happiest with the points and proposals/scientists 6, 5, 10 and 12 were fairly satisfied. 

- This is important to note as the last iteration placed more emphasis on happiness over discovery but was not able to fully achieve it. 

- Instead of trying to force happiness, lets try to understand why they aren't very happy and adjust the acquisition policy from there. 

- Important to note that happiness was found to be inversely proportional to scientific discovery and proportional to diversity.  

- Acquisition function for iteration 3 will be modified to put a higher weight on scientific discovery since emphasizing the community didn't result in higher than average happiness.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# ── 1. Load all new proposal CSVs ─────────────────────────────────────
PROPOSAL_DIR = "Proposals_iter2/"

proposal_files = sorted(glob.glob(os.path.join(PROPOSAL_DIR, "*.csv")))
print(f"Found {len(proposal_files)} proposal files:")
for f in proposal_files:
    print(f"  {os.path.basename(f)}")

# ── 2. Normalise column names ──────────────────────────────────────────
COL_MAP = {
    "Index"       : "Pool_index",
    "Prediction"  : "community_pred",
    "Uncertainty" : "community_uncert",
    "Score"       : "community_score",
    "pool_id"     : "Pool_index",
    "prediction"  : "community_pred",
    "std"         : "community_uncert",
    "score"       : "community_score",
}

proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp = tmp.rename(columns=COL_MAP)
    tmp["scientist_id"] = scientist_id
    tmp = tmp[["Pool_index", "community_pred", "community_uncert",
               "community_score", "scientist_id"]]
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"\nPooled: {proposals_pooled.shape[0]} rows from "
      f"{proposals_pooled['scientist_id'].nunique()} scientists")
print(proposals_pooled.groupby("scientist_id").size().to_string())

# ── 3. Join back to recover x1–x11 and true y ─────────────────────────
proposals_full = proposals_pooled.merge(
    df[["Pool_index"] + features + ["y"]],
    on="Pool_index",
    how="left"
)

# ── 4. Clean inf/nan in community scores ──────────────────────────────
proposals_full["community_score"] = proposals_full["community_score"].replace(
    [np.inf, -np.inf], np.nan
)
proposals_full["community_score"] = proposals_full.groupby("scientist_id")[
    "community_score"
].transform(lambda x: x.fillna(x.median()))
global_median = proposals_full["community_score"].median()
proposals_full["community_score"] = proposals_full["community_score"].fillna(global_median)

print(f"\nInf remaining : {np.isinf(proposals_full['community_score']).sum()}")
print(f"NaN remaining : {proposals_full['community_score'].isnull().sum()}")

# ── 5. Deduplicate ─────────────────────────────────────────────────────
proposals_deduped_iter3 = (
    proposals_full
    .groupby("Pool_index")
    .agg(
        community_score  = ("community_score", "mean"),
        scientist_id     = ("scientist_id",     lambda x: ",".join(sorted(set(x)))),
        community_pred   = ("community_pred",   "mean"),
        community_uncert = ("community_uncert", "mean"),
        **{f: (f, "first") for f in features},
        y                = ("y", "first"),
    )
    .reset_index()
)

print(f"\nUnique proposals after dedup: {len(proposals_deduped_iter3)}")

# ── 6. PCA for feature space visualisation ────────────────────────────
X_prop_scaled = scaler_X.transform(proposals_deduped_iter3[features].values)
pca           = PCA(n_components=2, random_state=1003)
X_prop_pca    = pca.fit_transform(X_prop_scaled)

# Also project full training pool for background reference
X_full_pca    = pca.transform(X_shuffled)

# ── 7. Assign a unique color per scientist ────────────────────────────
scientists    = proposals_full["scientist_id"].unique()
cmap          = plt.cm.get_cmap("tab20", len(scientists))
sci_color_map = {s: cmap(i) for i, s in enumerate(sorted(scientists))}

# Map each deduped point to its first scientist for colouring
proposals_deduped_iter3["primary_scientist"] = (
    proposals_deduped_iter3["scientist_id"].str.split(",").str[0]
)

# ── 8. Figure ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.suptitle("Iteration 3 — Community Proposals: What Are They Looking For?",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

ax_pca   = fig.add_subplot(gs[0, :2])   # PCA feature space
ax_pool  = fig.add_subplot(gs[0, 2])    # Pool_index distribution


# ─── Panel 1: PCA feature space ───────────────────────────────────────
ax_pca.scatter(X_full_pca[:, 0], X_full_pca[:, 1],
               c="lightgray", s=2, alpha=0.3, rasterized=True, label="Full pool")
for sci in sorted(scientists):
    mask = proposals_deduped_iter3["primary_scientist"] == sci
    ax_pca.scatter(
        X_prop_pca[mask, 0], X_prop_pca[mask, 1],
        color=sci_color_map[sci], s=50, edgecolors="black",
        linewidths=0.5, label=sci, zorder=5
    )
ax_pca.set_xlabel("PC1")
ax_pca.set_ylabel("PC2")
ax_pca.set_title("Feature Space Coverage\n(grey = full pool, colours = proposals by scientist)")
ax_pca.legend(fontsize=7, ncol=2, loc="upper right")
ax_pca.grid(True, alpha=0.2)

# ─── Panel 2: Pool_index distribution per scientist ───────────────────
for sci in sorted(scientists):
    mask    = proposals_full["scientist_id"] == sci
    indices = proposals_full.loc[mask, "Pool_index"].values
    ax_pool.scatter(
        indices,
        [sci] * len(indices),
        color=sci_color_map[sci], s=30,
        edgecolors="black", linewidths=0.4, alpha=0.8
    )
ax_pool.set_xlabel("Pool_index")
ax_pool.set_title("Pool Regions Targeted\nper Scientist")
ax_pool.tick_params(axis="y", labelsize=8)
ax_pool.grid(True, alpha=0.2, axis="x")



# ─── Helper: overlapping histograms ───────────────────────────────────
def hist_by_scientist(ax, col, xlabel, title):
    for sci in sorted(scientists):
        mask = proposals_full["scientist_id"] == sci
        vals = proposals_full.loc[mask, col].replace(
            [np.inf, -np.inf], np.nan
        ).dropna().values
        if len(vals) == 0:
            continue
        ax.hist(vals, bins=15, alpha=0.45, color=sci_color_map[sci],
                label=sci, edgecolor="none")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.2)



plt.show()


Now that we have a better understanding of what the community is looking for with their new experiments, lets use the acquistion function to score all the experiments from the scientists.

We can see clustering in the PCA space as well as in the index space to see which scientists in the community have similar goals (aka frenemies). 

Iteration 3 Acquisition Function:

$$S(x) = \alpha \cdot \text{UCB}_{\text{norm}}(x) + \beta \cdot d_{\text{norm}}(x) + \gamma \cdot \text{cs}_{\text{norm}}(x) + \delta \cdot \text{cov}_{\text{norm}}(x)$$

$$\alpha = 0.9, \quad \beta = 0.05, \quad \gamma = 0.025, \quad \delta = 0.025$$

In order to emphasize scientific discovery, the weight on the scientific discovery term $\alpha$ has been increased from 0.4 to 0.9 and the remaining community terms adjusted accordingly. 

This is because in the last acquisition function an emphasis was placed on the community terms but it did not result in a significantly higher than average happiness. 




In [ ]:
# ── Score all proposals using model_iter2 (before retraining) ─────────
X_prop_scaled = scaler_X.transform(proposals_deduped_iter3[features].values)

"""(S_prop3, mu_prop3, sigma_prop3,
 ucb_norm_prop3, d_norm_prop3,
 cs_norm_prop3, cov_norm_prop3,
 aligned_proposals3, aligned_pos3) = acquisition_score_iter2(
    model_iter2,
    proposals_deduped_iter3,
    pool_idx_shuf,
)"""

# Increased alpha (scientific discovery) decreased beta/gamma/delta
(S_prop3, mu_prop3, sigma_prop3,
 ucb_norm_prop3, d_norm_prop3,
 cs_norm_prop3, cov_norm_prop3,
 aligned_proposals3, aligned_pos3) = acquisition_score_iter2(
    model_iter2,
    proposals_deduped_iter3,
    pool_idx_shuf,
    alpha=0.9,   # up from 0.4 — scientific discovery
    beta=0.05,    # down from 0.3 — feature diversity
    gamma=0.025,   # down from 0.2 — community score
    delta=0.025,   # down from 0.1 — scientist coverage
)


print(f"Scored {len(S_prop3)} unique proposal points using model_iter2")
print(f"Score range: [{S_prop3.min():.4f}, {S_prop3.max():.4f}]")

In [ ]:
# ── Select top 20 from proposals ──────────────────────────────────────
top20_local3 = np.argsort(S_prop3)[::-1][:20]
top20_pos3   = aligned_pos3[top20_local3]
top20_pool3  = pool_idx_shuf[top20_pos3]

results_iter3 = aligned_proposals3.iloc[top20_local3].copy().reset_index(drop=True)
results_iter3["y_pred"]        = mu_prop3[top20_local3]
results_iter3["y_uncert"]      = sigma_prop3[top20_local3]
results_iter3["ucb_norm"]      = ucb_norm_prop3[top20_local3]
results_iter3["d_norm"]        = d_norm_prop3[top20_local3]
results_iter3["cs_norm"]       = cs_norm_prop3[top20_local3]
results_iter3["coverage_norm"] = cov_norm_prop3[top20_local3]
results_iter3["score"]         = S_prop3[top20_local3]
top20_pos_plot3                = aligned_pos3[top20_local3]

scientist_counts = results_iter3["scientist_id"].value_counts()
print(f"Top 20 selected — scientists represented:\n{scientist_counts.to_string()}")
print(f"\nTop 20 Pool_index values: {top20_pool3}")
print(results_iter3[["Pool_index", "scientist_id",
                      "community_score", "score"]].to_string(index=False))

In [ ]:
# ── Add only the selected 20 points to the training set ───────────────
selected_proposals = proposals_deduped_iter3[
    proposals_deduped_iter3["Pool_index"].isin(top20_pool3)
]

# Exclude any already in the current training set
train_pool_indices3 = set(pool_idx_shuf2[train_idx])
new_selected_mask   = ~selected_proposals["Pool_index"].isin(train_pool_indices3)
new_selected        = selected_proposals[new_selected_mask]

X_new3 = scaler_X.transform(new_selected[features].values)
y_new3 = scaler_y.transform(new_selected["y"].values.reshape(-1, 1)).ravel()

X_train_aug3 = np.vstack([X_train_aug2, X_new3])
y_train_aug3 = np.concatenate([y_train_aug2, y_new3])

print(f"Previous training set : {X_train_aug2.shape[0]} points")
print(f"New selected points   : {X_new3.shape[0]} points added")
print(f"New training set      : {X_train_aug3.shape[0]} points")

In [ ]:
# ── Retrain surrogate on augmented training set — Iteration 3 ──────────
X_pool_aug3    = np.vstack([X_aug_shuf2, X_new3])
y_pool_aug3    = np.concatenate([y_aug_shuf2, y_new3])
pool_idx_shuf3 = np.concatenate([
    pool_idx_shuf2,
    new_selected["Pool_index"].values
])

np.random.seed(1003)
shuffle_idx3   = np.random.permutation(len(y_pool_aug3))
X_aug_shuf3    = X_pool_aug3[shuffle_idx3]
y_aug_shuf3    = y_pool_aug3[shuffle_idx3]
pool_idx_shuf3 = pool_idx_shuf3[shuffle_idx3]

model_iter3 = SurrogateGP(random_state=1003)
model_iter3.fit(X_train_aug3, y_train_aug3)

model_iter3.X_scaled_ = X_aug_shuf3
model_iter3.y_        = y_aug_shuf3

print(f"model_iter3 fitted on  : {X_train_aug3.shape[0]} points")
print(f"Scoring pool           : {X_aug_shuf3.shape[0]} points")

In [ ]:
S3, mu3, sigma3 = model_iter3.score()
d3, X_pca3      = objective_community_engagement(
    model_iter3.X_scaled_, n_components=model_iter3.n_components, k=model_iter3.k_neighbors
)
ucb3      = mu3 + model_iter3.kappa * sigma3
ucb_norm3 = (ucb3 - ucb3.min()) / (ucb3.max() - ucb3.min())
d_norm3   = (d3   - d3.min())   / (d3.max()   - d3.min())

# Pareto on top 500 by score
objectives3 = np.stack([ucb_norm3, d_norm3], axis=1)
top500_idx3 = np.argsort(S3)[::-1][:500]
is_pareto3  = is_non_dominated(objectives3[top500_idx3])
pareto_idx3 = top500_idx3[is_pareto3]

fig = plt.figure(figsize=(16, 11))
fig.suptitle("Iteration 3 — Acquisition Policy: Pareto Frontier & Objective Space (α=0.6)", 
#fig.suptitle("Iteration 3 — Acquisition Policy: Pareto Frontier & Objective Space",
             fontsize=14, fontweight="bold", y=0.99)

gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
ax_s = fig.add_subplot(gs[0, :])
ax_o = fig.add_subplot(gs[1, 0])
ax_p = fig.add_subplot(gs[1, 1])

# ─── Panel 1: Acquisition score across augmented pool ─────────────────
ax_s.scatter(np.arange(len(S3)), S3,
             c=S3, cmap="viridis", s=2, alpha=0.3, rasterized=True)
ax_s.scatter(top20_pos3, S3[top20_pos3],
             c="red", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_s.scatter(pareto_idx3, S3[pareto_idx3],
             c="gold", s=40, zorder=4, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated (top 500)")
ax_s.set_xlabel("Pool index")
ax_s.set_ylabel("Acquisition score S(x)")
ax_s.set_title("Acquisition Score Across Augmented Pool")
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.2)

# ─── Panel 2: Objective space with Pareto frontier ────────────────────
ax_o.scatter(ucb_norm3, d_norm3,
             c="steelblue", s=3, alpha=0.15, rasterized=True, label="All points")
ax_o.scatter(ucb_norm3[pareto_idx3], d_norm3[pareto_idx3],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, marker="*", label="Pareto-non-dominated")
ax_o.scatter(ucb_norm3[top20_pos3], d_norm3[top20_pos3],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
pareto_pts3 = np.stack([ucb_norm3[pareto_idx3], d_norm3[pareto_idx3]], axis=1)
sorted_p3   = pareto_pts3[np.argsort(pareto_pts3[:, 0])]
ax_o.step(sorted_p3[:, 0], sorted_p3[:, 1], where="post",
          color="gold", lw=2, ls="--", alpha=0.85, zorder=4, label="Pareto front")
ax_o.set_xlabel("UCB$_{norm}$(x) — Scientific Discovery")
ax_o.set_ylabel("$d_{norm}$(x) — Community Engagement")
ax_o.set_title("Objective Space with Pareto Frontier")
ax_o.legend(fontsize=9)
ax_o.grid(True, alpha=0.2)

# ─── Panel 3: PCA space ───────────────────────────────────────────────
sc = ax_p.scatter(X_pca3[:, 0], X_pca3[:, 1],
                  c=S3, cmap="viridis", s=3, alpha=0.3, rasterized=True)
ax_p.scatter(X_pca3[pareto_idx3, 0], X_pca3[pareto_idx3, 1],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated")
ax_p.scatter(X_pca3[top20_pos3, 0], X_pca3[top20_pos3, 1],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_p.set_xlabel("PC1")
ax_p.set_ylabel("PC2")
ax_p.set_title("PCA Feature Space\n(colour = acquisition score)")
ax_p.legend(fontsize=9)
ax_p.grid(True, alpha=0.2)
plt.colorbar(sc, ax=ax_p, label="S(x)")

plt.show()

A good distribution of points in the PCA-space were selected, however no non dominated points were selected. This could be due to the acquisition function but it could also be a carry over affect from the last iteration. The scientists might have adjusted their proposals for diversity and suggested less ambitious points. This gives a smaller pool of points that are pushing the pareto envelope. 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_tr_eval3, X_te_eval3, y_tr_eval3, y_te_eval3 = train_test_split(
    X_train_aug3, y_train_aug3, test_size=0.2, random_state=1003
)

gp_eval3 = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval3.fit(X_tr_eval3, y_tr_eval3)
mu_test3, sigma_test3 = gp_eval3.predict(X_te_eval3, return_std=True)

rmse3     = np.sqrt(mean_squared_error(y_te_eval3, mu_test3))
r2_3      = 1 - np.sum((y_te_eval3 - mu_test3)**2) / np.sum((y_te_eval3 - y_te_eval3.mean())**2)
mean_std3 = sigma_test3.mean()

print(f"Test RMSE : {rmse3:.4f}")
print(f"Test R²   : {r2_3:.4f}")
print(f"Mean σ    : {mean_std3:.4f}")

fig = plt.figure(figsize=(18, 11))
fig.suptitle("Iteration 3 — Surrogate Performance, Uncertainty & Acquisition Policy (α=0.6)",
#fig.suptitle("Iteration 3 — Surrogate Performance, Uncertainty & Acquisition Policy",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit = fig.add_subplot(gs[0, :2])
ax_res = fig.add_subplot(gs[0, 2:])
ax_ucb = fig.add_subplot(gs[1, 0])
ax_std = fig.add_subplot(gs[1, 1])
ax_div = fig.add_subplot(gs[1, 2])
ax_s   = fig.add_subplot(gs[1, 3])

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_te_eval3, mu_test3, c=sigma_test3, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_te_eval3.min(), mu_test3.min()), max(y_te_eval3.max(), mu_test3.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse3:.4f}  R²={r2_3:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test3.min(), sigma_test3.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals3 = y_te_eval3 - mu_test3
ax_res.scatter(mu_test3, residuals3, c=sigma_test3, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test3.min(), mu_test3.max(), 100),
    -2 * sigma_test3.mean(), 2 * sigma_test3.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std3:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper ───────────────────────────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 20 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6 ───────────────────────────────────────────────────────
dist_panel(ax_ucb, ucb_norm3, ucb_norm3[top20_pos3],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution",       "#3498db")
dist_panel(ax_std, sigma3,    sigma3[top20_pos3],
           "σ(x)",            "Model Uncertainty\nσ distribution",             "#9b59b6")
dist_panel(ax_div, d_norm3,   d_norm3[top20_pos3],
           "$d_{norm}$(x)",   "Community Engagement\nDiversity distribution",  "#2ecc71")
dist_panel(ax_s,   S3,        S3[top20_pos3],
           "S(x)",            "Combined Acquisition\nScore S(x)",              "#e67e22")

plt.show()


Test RMSE and R2 increased again with integration of the selected points in the training set, as expected. 

The mean of the selected points have less model uncertainty than the last iteration. This means that the mean is driving the scientific discovery UCB term.

In [ ]:
# ── Save full results for future reference ─────────────────────────────
results_iter3_save = results_iter3.copy()

feat_rename = {features[i]: f"x{i+1}" for i in range(11)}
results_iter3_save = results_iter3_save.rename(columns=feat_rename)
results_iter3_save["sis_id"] = "ansarir5"

save_cols = (
    ["sis_id", "Pool_index"]
    + [f"x{i}" for i in range(1, 12)]
    + ["y", "score", "ucb_norm", "d_norm", "cs_norm",
       "coverage_norm", "y_pred", "y_uncert", "scientist_id", "community_score"]
)

results_iter3_save[save_cols].to_csv("iteration3_results_full.csv", index=False)
print("Saved iteration3_results_full.csv")

# ── Save submission CSV ────────────────────────────────────────────────
def make_explanation_iter3(row):
    feat_cols = [f"x{i}" for i in range(1, 12)]
    top_feat  = row[feat_cols].abs().idxmax()
    n_scientists = len(row["scientist_id"].split(","))
    return (
        f"Selected via multi-objective acquisition (S={row['score']:.3f}) "
        f"balancing scientific discovery (UCB: mu={row['y_pred']:.3f}, sigma={row['y_uncert']:.3f}) "
        f"and community engagement (diversity={row['d_norm']:.3f}, "
        f"community_score={row['community_score']:.3f}, "
        f"proposed by {n_scientists} scientist(s): {row['scientist_id']}). "
        f"Most dominant feature: {top_feat}={row[top_feat]:.3f}."
    )

submission_iter3 = results_iter3_save.copy()
submission_iter3["explanation"] = submission_iter3.apply(make_explanation_iter3, axis=1)

feat_cols = [f"x{i}" for i in range(1, 12)]
submission_iter3 = submission_iter3[
    ["sis_id", "Pool_index"] + feat_cols + ["y", "score", "explanation"]
]

submission_iter3 = submission_iter3.sort_values("score", ascending=False).reset_index(drop=True)

print(submission_iter3[["sis_id", "Pool_index", "y", "score"]].to_string(index=False))

submission_iter3.to_csv("iteration3_submission.csv", index=False)
print("\nSaved iteration3_submission.csv")

In [ ]:
submission_iter3

### Iteration 4

In [ ]:
# ── Parse happiness scores from text file ──────────────────────────────
happiness_raw3 = open("Proposals_iter3/happiness_results.txt").read()

happiness3 = {}
lines = happiness_raw3.strip().split("\n")
current_proposal = None
for line in lines:
    line = line.strip()
    if line.startswith("Proposal file:"):
        current_proposal = line.split("Proposal file:")[-1].strip().replace(".csv", "")
    elif line.startswith("Happiness with samples selected"):
        score = float(line.split(":")[-1].strip())
        happiness3[current_proposal] = score

print("Parsed happiness scores:")
for k, v in sorted(happiness3.items()):
    print(f"  {k}: {v:.4f}")

In [ ]:
#comparing iteration2_submission and happiness_results

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd


# ── Map happiness back to each selected point ──────────────────────────
# Points proposed by multiple scientists get the mean happiness
def mean_happiness(scientist_id_str):
    scientists = scientist_id_str.split(",")
    return np.mean([happiness3[s.strip()] for s in scientists])

results_iter3["happiness"] = results_iter3["scientist_id"].apply(mean_happiness)


# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Iteration 3 — Top 20 Selected Points vs Community Happiness",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

ax_bar  = fig.add_subplot(gs[0, :2])   # happiness per selected point
ax_sci  = fig.add_subplot(gs[0, 2])    # mean happiness per scientist
ax_sco  = fig.add_subplot(gs[1, 0])    # happiness vs acquisition score
ax_ucb  = fig.add_subplot(gs[1, 1])    # happiness vs UCB
ax_div  = fig.add_subplot(gs[1, 2])    # happiness vs diversity

# ─── Panel 1: Happiness per selected point ────────────────────────────
colors = plt.cm.RdYlGn(
    (results_iter3["happiness"] - 1) / 4
)
bars = ax_bar.bar(
    range(len(results_iter3)),
    results_iter3["happiness"],
    color=colors, edgecolor="black", linewidth=0.6
)
ax_bar.axhline(results_iter3["happiness"].mean(), color="black",
               lw=1.5, ls="--",
               label=f"Mean happiness = {results_iter3['happiness'].mean():.2f}")
ax_bar.set_xticks(range(len(results_iter3)))
ax_bar.set_xticklabels(results_iter3["Pool_index"], rotation=45, ha="right", fontsize=8)
ax_bar.set_xlabel("Pool_index")
ax_bar.set_ylabel("Happiness score")
ax_bar.set_title("Happiness Score per Selected Point\n(colour: red=low, green=high)")
ax_bar.set_ylim(0, 5.5)
ax_bar.legend(fontsize=9)
ax_bar.grid(True, alpha=0.2, axis="y")

# ─── Panel 2: Mean happiness per scientist ────────────────────────────
sci_df = pd.DataFrame(happiness3.items(), columns=["scientist_id", "happiness"])
sci_df = sci_df.sort_values("happiness", ascending=True)
bar_colors = plt.cm.RdYlGn((sci_df["happiness"] - 1) / 4)
ax_sci.barh(sci_df["scientist_id"], sci_df["happiness"],
            color=bar_colors, edgecolor="black", linewidth=0.6)
ax_sci.axvline(sci_df["happiness"].mean(), color="black", lw=1.5, ls="--",
               label=f"Mean = {sci_df['happiness'].mean():.2f}")
ax_sci.set_xlabel("Happiness score")
ax_sci.set_title("Happiness per Scientist")
ax_sci.set_xlim(0, 5.5)
ax_sci.legend(fontsize=9)
ax_sci.grid(True, alpha=0.2, axis="x")

# ─── Helper: scatter with correlation ─────────────────────────────────
def scatter_corr(ax, x, y, xlabel, title, color):
    ax.scatter(x, y, c=color, s=60, edgecolors="black", linewidths=0.6, alpha=0.8)
    # correlation
    r = np.corrcoef(x, y)[0, 1]
    m, b = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m * xline + b, "k--", lw=1.5, label=f"r = {r:.2f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Happiness")
    ax.set_title(title)
    ax.set_ylim(0, 5.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

# ─── Panels 3–5: Happiness vs score components ────────────────────────
scatter_corr(ax_sco, results_iter3["score"],    results_iter3["happiness"],
             "Acquisition score S(x)",  "Happiness vs Acquisition Score", "#e67e22")
scatter_corr(ax_ucb, results_iter3["ucb_norm"], results_iter3["happiness"],
             "UCB$_{norm}$(x)",         "Happiness vs Scientific Discovery", "#3498db")
scatter_corr(ax_div, results_iter3["d_norm"],   results_iter3["happiness"],
             "$d_{norm}$(x)",           "Happiness vs Diversity", "#2ecc71")

plt.show()

- We can see that mean happiness went up to 3.48 from 2.57 meaning overall the community increased in happiness despite community emphasis not being our goal. Perhaps they just really like science and we should not overcomplicate our acquisition function. 

- Overall proposals/scientists 10 was the happiest with the points and proposals/scientists 3, 2, and 5 were fairly disappointed. 

- Happiness was not found to be correlated with scientific discovery or diversity or the acquisition score as a whole.

- The acquisition function will be modified to better capture the relationship to happiness. 

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# ── 1. Load all new proposal CSVs ─────────────────────────────────────
PROPOSAL_DIR = "Proposals_iter3/"

proposal_files = sorted(glob.glob(os.path.join(PROPOSAL_DIR, "*.csv")))
print(f"Found {len(proposal_files)} proposal files:")
for f in proposal_files:
    print(f"  {os.path.basename(f)}")

# ── 2. Normalise column names ──────────────────────────────────────────
COL_MAP = {
    "Index"       : "Pool_index",
    "Prediction"  : "community_pred",
    "Uncertainty" : "community_uncert",
    "Score"       : "community_score",
    "pool_id"     : "Pool_index",
    "prediction"  : "community_pred",
    "std"         : "community_uncert",
    "score"       : "community_score",
}

proposal_dfs = []
for fpath in proposal_files:
    scientist_id = os.path.basename(fpath).replace(".csv", "")
    tmp = pd.read_csv(fpath)
    tmp = tmp.rename(columns=COL_MAP)
    tmp["scientist_id"] = scientist_id
    tmp = tmp[["Pool_index", "community_pred", "community_uncert",
               "community_score", "scientist_id"]]
    proposal_dfs.append(tmp)

proposals_pooled = pd.concat(proposal_dfs, ignore_index=True)
proposals_pooled = proposals_pooled.dropna(subset=["Pool_index"])
proposals_pooled["Pool_index"] = proposals_pooled["Pool_index"].astype(int)

print(f"\nPooled: {proposals_pooled.shape[0]} rows from "
      f"{proposals_pooled['scientist_id'].nunique()} scientists")
print(proposals_pooled.groupby("scientist_id").size().to_string())

# ── 3. Join back to recover x1–x11 and true y ─────────────────────────
proposals_full = proposals_pooled.merge(
    df[["Pool_index"] + features + ["y"]],
    on="Pool_index",
    how="left"
)

# ── 4. Clean inf/nan in community scores ──────────────────────────────
proposals_full["community_score"] = proposals_full["community_score"].replace(
    [np.inf, -np.inf], np.nan
)
proposals_full["community_score"] = proposals_full.groupby("scientist_id")[
    "community_score"
].transform(lambda x: x.fillna(x.median()))
global_median = proposals_full["community_score"].median()
proposals_full["community_score"] = proposals_full["community_score"].fillna(global_median)

print(f"\nInf remaining : {np.isinf(proposals_full['community_score']).sum()}")
print(f"NaN remaining : {proposals_full['community_score'].isnull().sum()}")

# ── 5. Deduplicate ─────────────────────────────────────────────────────
proposals_deduped_iter4 = (
    proposals_full
    .groupby("Pool_index")
    .agg(
        community_score  = ("community_score", "mean"),
        scientist_id     = ("scientist_id",     lambda x: ",".join(sorted(set(x)))),
        community_pred   = ("community_pred",   "mean"),
        community_uncert = ("community_uncert", "mean"),
        **{f: (f, "first") for f in features},
        y                = ("y", "first"),
    )
    .reset_index()
)

print(f"\nUnique proposals after dedup: {len(proposals_deduped_iter4)}")

# ── 6. PCA for feature space visualisation ────────────────────────────
X_prop_scaled = scaler_X.transform(proposals_deduped_iter4[features].values)
pca           = PCA(n_components=2, random_state=1003)
X_prop_pca    = pca.fit_transform(X_prop_scaled)

# Also project full training pool for background reference
X_full_pca    = pca.transform(X_shuffled)

# ── 7. Assign a unique color per scientist ────────────────────────────
scientists    = proposals_full["scientist_id"].unique()
cmap          = plt.cm.get_cmap("tab20", len(scientists))
sci_color_map = {s: cmap(i) for i, s in enumerate(sorted(scientists))}

# Map each deduped point to its first scientist for colouring
proposals_deduped_iter4["primary_scientist"] = (
    proposals_deduped_iter4["scientist_id"].str.split(",").str[0]
)

# ── 8. Figure ──────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.suptitle("Iteration 4 — Community Proposals: What Are They Looking For?",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

ax_pca   = fig.add_subplot(gs[0, :2])   # PCA feature space
ax_pool  = fig.add_subplot(gs[0, 2])    # Pool_index distribution


# ─── Panel 1: PCA feature space ───────────────────────────────────────
ax_pca.scatter(X_full_pca[:, 0], X_full_pca[:, 1],
               c="lightgray", s=2, alpha=0.3, rasterized=True, label="Full pool")
for sci in sorted(scientists):
    mask = proposals_deduped_iter4["primary_scientist"] == sci
    ax_pca.scatter(
        X_prop_pca[mask, 0], X_prop_pca[mask, 1],
        color=sci_color_map[sci], s=50, edgecolors="black",
        linewidths=0.5, label=sci, zorder=5
    )
ax_pca.set_xlabel("PC1")
ax_pca.set_ylabel("PC2")
ax_pca.set_title("Feature Space Coverage\n(grey = full pool, colours = proposals by scientist)")
ax_pca.legend(fontsize=7, ncol=2, loc="upper right")
ax_pca.grid(True, alpha=0.2)

# ─── Panel 2: Pool_index distribution per scientist ───────────────────
for sci in sorted(scientists):
    mask    = proposals_full["scientist_id"] == sci
    indices = proposals_full.loc[mask, "Pool_index"].values
    ax_pool.scatter(
        indices,
        [sci] * len(indices),
        color=sci_color_map[sci], s=30,
        edgecolors="black", linewidths=0.4, alpha=0.8
    )
ax_pool.set_xlabel("Pool_index")
ax_pool.set_title("Pool Regions Targeted\nper Scientist")
ax_pool.tick_params(axis="y", labelsize=8)
ax_pool.grid(True, alpha=0.2, axis="x")


# ─── Helper: overlapping histograms ───────────────────────────────────
def hist_by_scientist(ax, col, xlabel, title):
    for sci in sorted(scientists):
        mask = proposals_full["scientist_id"] == sci
        vals = proposals_full.loc[mask, col].replace(
            [np.inf, -np.inf], np.nan
        ).dropna().values
        if len(vals) == 0:
            continue
        ax.hist(vals, bins=15, alpha=0.45, color=sci_color_map[sci],
                label=sci, edgecolor="none")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.2)



plt.show()


The scientists are looking for a much more diverse range of proposals in terms of coverage. By pool index there seems to be a high concentration in indicies from 3000 to 6000.

For iteration 4, a more principled scientific discovery objective was introduced: Expected Hypervolume Improvement.

EHVI asks a fundamentally different question from UCB. Instead of optimizing a balance of predicted mean and uncertainty, it directly estimates: how much do we expect this point to improve upon the best result we've seen so far? It does this via 5,000 Monte Carlo samples from the GP posterior N(μ(x), σ²(x)), computing the average positive improvement over y*. This is more targeted for later-stage optimization when we have enough data to have confidence in the current best.

$$S(x) = \alpha \cdot \text{EHVI}_{\text{norm}}(x) + \beta \cdot \text{UCB}_{\text{norm}}(x) + \gamma \cdot d_{\text{norm}}(x) + \delta \cdot \text{region}_{\text{norm}}(x)$$

$$\alpha = 0.3, \quad \beta = 0.3, \quad \gamma = 0.2, \quad \delta = 0.2, \quad \sum = 1.0$$

$$\text{EHVI}(x) = \mathbb{E}\left[\max(0, \, y - y^*)\right], \quad y \sim \mathcal{N}(\mu(x), \sigma^2(x)), \quad y^* = \max(y_{\text{train}})$$

$$\text{UCB}(x) = \mu(x) + \kappa \cdot \sigma(x), \quad \kappa = 2.0$$

$$\text{region}(x) = \frac{1}{n_b + 1}, \quad n_b = \text{count of proposals in Pool\_index bin of } x, \quad n_{\text{bins}} = 10$$

I also replaced the scientist coverage term with a Pool_index region diversity term, binning the proposal space into 10 equal-width index ranges and scoring points by the inverse count of proposals in that bin. This rewards points from underrepresented index ranges, ensuring selections span the full proposal pool rather than clustering in one region.

The community scientist score was dropped entirely. The happiness analysis showed it wasn't improving satisfaction. I removed it and focused on understanding why scientists are unhappy from the structural properties of the design space.

In [ ]:
def compute_ehvi(mu, sigma, current_best, n_samples=1000, random_state=1003):
    """
    Monte Carlo approximation of Expected Hypervolume Improvement.

    For each candidate point x, samples from N(mu(x), sigma(x)) and
    computes the expected improvement over the current best observed y.

    Parameters
    ----------
    mu           : np.ndarray (n_points,) — GP posterior means
    sigma        : np.ndarray (n_points,) — GP posterior std deviations
    current_best : float — best observed y value in current training set
    n_samples    : int   — Monte Carlo samples per point

    Returns
    -------
    ehvi         : np.ndarray (n_points,) — expected hypervolume improvement
    """
    np.random.seed(random_state)
    ehvi = np.zeros(len(mu))
    for i in range(len(mu)):
        samples      = np.random.normal(mu[i], sigma[i], n_samples)
        improvements = np.maximum(0, samples - current_best)
        ehvi[i]      = improvements.mean()
    return ehvi


def acquisition_score_iter4(
    model,
    proposals,
    pool_idx_shuf,
    alpha=0.3,    # EHVI — Pareto frontier improvement
    beta=0.3,     # UCB  — high predicted y + uncertainty
    gamma=0.2,    # feature-space diversity (k-NN in PCA space)
    delta=0.2,    # pool_id region diversity
    kappa=2.0,
    n_components=5,
    k=20,
    n_samples=5000,
    n_bins=10,    # number of Pool_index bins for region diversity
):
    """
    Iteration 5 acquisition score.

    S(x) = alpha * EHVI_norm(x)
         + beta  * UCB_norm(x)
         + gamma * d_norm(x)        — feature-space k-NN diversity
         + delta * region_norm(x)   — Pool_index region diversity

    Community scientist score and coverage are replaced by Pool_index
    region diversity — rewards points from underrepresented index ranges,
    ensuring selections span the full proposal space rather than
    clustering in a single region.

    Parameters
    ----------
    model         : fitted SurrogateGP
    proposals     : pd.DataFrame — deduplicated proposals
    pool_idx_shuf : np.ndarray — Pool_index values aligned to model.X_scaled_
    alpha         : weight for EHVI
    beta          : weight for UCB
    gamma         : weight for feature-space diversity
    delta         : weight for Pool_index region diversity
    kappa         : UCB exploration parameter
    n_samples     : Monte Carlo samples for EHVI
    n_bins        : number of Pool_index bins for region scoring
    """
    assert np.isclose(alpha + beta + gamma + delta, 1.0), "weights must sum to 1"

    # ── Locate proposal rows in augmented pool ─────────────────────────
    proposal_pool_indices = proposals["Pool_index"].values
    mask          = np.isin(pool_idx_shuf, proposal_pool_indices)
    proposal_rows = np.where(mask)[0]
    idx_map       = {pid: pos for pos, pid in zip(proposal_rows,
                     pool_idx_shuf[proposal_rows])}
    aligned_pos   = np.array([idx_map[pid] for pid in proposal_pool_indices
                               if pid in idx_map])
    aligned       = proposals[
        proposals["Pool_index"].isin(pool_idx_shuf[aligned_pos])
    ].copy().reset_index(drop=True)

    X_prop = model.X_scaled_[aligned_pos]

    # ── GP predictions ─────────────────────────────────────────────────
    mu, sigma = model.gp_.predict(X_prop, return_std=True)

    # ── Objective 1: EHVI ─────────────────────────────────────────────
    current_best = model.y_.max()
    ehvi         = compute_ehvi(mu, sigma, current_best, n_samples=n_samples)
    ehvi_norm    = (ehvi - ehvi.min()) / (ehvi.max() - ehvi.min() + 1e-8)

    # ── Objective 2: UCB ──────────────────────────────────────────────
    ucb      = mu + kappa * sigma
    ucb_norm = (ucb - ucb.min()) / (ucb.max() - ucb.min() + 1e-8)

    # ── Objective 3: Feature-space diversity (k-NN) ───────────────────
    d, _ = objective_community_engagement(
        X_prop, n_components=n_components, k=min(k, len(X_prop) - 1)
    )
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)

    # ── Objective 4: Pool_index region diversity ───────────────────────
    # Bin the full Pool_index range into n_bins equal-width buckets.
    # Count how many proposals fall in each bin — bins with fewer proposals
    # are underrepresented and get a higher diversity score.
    # This rewards points from sparse Pool_index regions (e.g. 8000–12000).
    pool_indices  = aligned["Pool_index"].values
    bin_edges     = np.linspace(pool_indices.min(), pool_indices.max() + 1, n_bins + 1)
    bin_assign    = np.digitize(pool_indices, bin_edges) - 1
    bin_assign    = np.clip(bin_assign, 0, n_bins - 1)
    bin_counts    = np.bincount(bin_assign, minlength=n_bins)

    # Points in less-populated bins get higher scores
    # Add 1 to avoid division by zero
    region_score  = 1.0 / (bin_counts[bin_assign] + 1)
    region_norm   = (region_score - region_score.min()) / (
                     region_score.max() - region_score.min() + 1e-8)

    # ── Combined score ────────────────────────────────────────────────
    S = (alpha * ehvi_norm
       + beta  * ucb_norm
       + gamma * d_norm
       + delta * region_norm)

    return S, mu, sigma, ehvi_norm, ucb_norm, d_norm, region_norm, aligned, aligned_pos



In [ ]:
(S_prop4, mu_prop4, sigma_prop4,
 ehvi_norm_prop4, ucb_norm_prop4,
 d_norm_prop4, region_norm_prop4,
 aligned_proposals4, aligned_pos4) = acquisition_score_iter4(
    model_iter3,
    proposals_deduped_iter4,
    pool_idx_shuf3,
    alpha=0.45,
    beta=0.10,
    gamma=0.30,
    delta=0.15,
    n_bins=10,
    n_samples=5000,
)

"""Parameters
    ----------
    model         : fitted SurrogateGP
    proposals     : pd.DataFrame — deduplicated proposals
    pool_idx_shuf : np.ndarray — Pool_index values aligned to model.X_scaled_
    alpha         : weight for EHVI
    beta          : weight for UCB
    gamma         : weight for feature-space diversity
    delta         : weight for Pool_index region diversity
    kappa         : UCB exploration parameter
    n_samples     : Monte Carlo samples for EHVI
    n_bins        : number of Pool_index bins for region scoring"""


print(f"Scored {len(S_prop4)} unique proposal points using model_iter3")
print(f"Score range: [{S_prop4.min():.4f}, {S_prop4.max():.4f}]")

In [ ]:
# ── Select top 20 from proposals ──────────────────────────────────────
top20_local4 = np.argsort(S_prop4)[::-1][:20]
top20_pos4   = aligned_pos4[top20_local4]
top20_pool4  = pool_idx_shuf3[top20_pos4]

results_iter4 = aligned_proposals4.iloc[top20_local4].copy().reset_index(drop=True)
results_iter4["y_pred"]        = mu_prop4[top20_local4]
results_iter4["y_uncert"]      = sigma_prop4[top20_local4]
results_iter4["ucb_norm"]      = ucb_norm_prop4[top20_local4]
results_iter4["d_norm"]        = d_norm_prop4[top20_local4]
#results_iter4["cs_norm"]       = cs_norm_prop4[top20_local4]
#results_iter4["coverage_norm"] = cov_norm_prop4[top20_local4]
results_iter4["score"]         = S_prop4[top20_local4]
top20_pos_plot4                = aligned_pos4[top20_local4]

scientist_counts = results_iter4["scientist_id"].value_counts()
print(f"Top 20 selected — scientists represented:\n{scientist_counts.to_string()}")
print(f"\nTop 20 Pool_index values: {top20_pool4}")
print(results_iter4[["Pool_index", "scientist_id",
                      "community_score", "score"]].to_string(index=False))

In [ ]:
# ── Add only the selected 20 points to the training set ───────────────
selected_proposals4 = proposals_deduped_iter4[
    proposals_deduped_iter4["Pool_index"].isin(top20_pool4)
]

# Use the actual Pool_index values in the iter 3 training set
# pool_idx_shuf3 aligned to X_aug_shuf3 — take all indices in X_train_aug3
train_pool_indices4 = set(
    np.concatenate([
        training_shuffled.iloc[train_idx]["Pool_index"].values,  # only the 3000 sampled points
        new_selected2["Pool_index"].values,                       # iter 2 additions
        new_selected["Pool_index"].values,                        # iter 3 additions
    ])
)

new_selected_mask4 = ~selected_proposals4["Pool_index"].isin(train_pool_indices4)
new_selected4      = selected_proposals4[new_selected_mask4]

if len(new_selected4) > 0:
    X_new4       = scaler_X.transform(new_selected4[features].values)
    y_new4       = scaler_y.transform(new_selected4["y"].values.reshape(-1, 1)).ravel()
    X_train_aug4 = np.vstack([X_train_aug3, X_new4])
    y_train_aug4 = np.concatenate([y_train_aug3, y_new4])
    print(f"New points added      : {len(new_selected4)}")
else:
    X_new4       = np.empty((0, X_train_aug3.shape[1]))
    y_new4       = np.array([])
    X_train_aug4 = X_train_aug3.copy()
    y_train_aug4 = y_train_aug3.copy()
    print("No new points to add — all selected proposals already in training set")

print(f"Previous training set : {X_train_aug3.shape[0]} points")
print(f"New training set      : {X_train_aug4.shape[0]} points")



In [ ]:
print(f"Total points in training set : {len(train_pool_indices4)}")
print(f"top20_pool4 values           : {top20_pool4}")
print(f"Total already in training    : {sum(p in train_pool_indices4 for p in top20_pool4)}/20")
print(f"Pool_index values NOT in training set:")
print([p for p in top20_pool4 if p not in train_pool_indices4])


In [ ]:
top20_pool4

In [ ]:
print(f"top20_pool4 values       : {top20_pool4}")
print(f"\nIn training_shuffled     : {sum(p in set(training_shuffled['Pool_index'].values) for p in top20_pool4)}/20")
print(f"In new_selected2         : {sum(p in set(new_selected2['Pool_index'].values) for p in top20_pool4)}/20")
print(f"In new_selected (iter3)  : {sum(p in set(new_selected['Pool_index'].values) for p in top20_pool4)}/20")
print(f"\nTotal already in training: {sum(p in train_pool_indices4 for p in top20_pool4)}/20")
print(f"\nPool_index values NOT in training set:")
print([p for p in top20_pool4 if p not in train_pool_indices4])

In [ ]:
# ── Retrain surrogate on augmented training set — Iteration 4 ──────────
X_pool_aug4    = np.vstack([X_aug_shuf3, X_new4])
y_pool_aug4    = np.concatenate([y_aug_shuf3, y_new4])
pool_idx_shuf4 = np.concatenate([
    pool_idx_shuf3,
    new_selected4["Pool_index"].values
])

np.random.seed(1003)
shuffle_idx4   = np.random.permutation(len(y_pool_aug4))
X_aug_shuf4    = X_pool_aug4[shuffle_idx4]
y_aug_shuf4    = y_pool_aug4[shuffle_idx4]
pool_idx_shuf4 = pool_idx_shuf4[shuffle_idx4]

model_iter4 = SurrogateGP(random_state=1003)
model_iter4.fit(X_train_aug4, y_train_aug4)

model_iter4.X_scaled_ = X_aug_shuf4
model_iter4.y_        = y_aug_shuf4

print(f"model_iter4 fitted on  : {X_train_aug4.shape[0]} points")
print(f"Scoring pool           : {X_aug_shuf4.shape[0]} points")

In [ ]:
S4, mu4, sigma4 = model_iter4.score()
d4, X_pca4      = objective_community_engagement(
    model_iter4.X_scaled_, n_components=model_iter4.n_components, k=model_iter4.k_neighbors
)
ucb4      = mu4 + model_iter4.kappa * sigma4
ucb_norm4 = (ucb4 - ucb4.min()) / (ucb4.max() - ucb4.min())
d_norm4   = (d4   - d4.min())   / (d4.max()   - d4.min())

# Pareto on top 500 by score
objectives4 = np.stack([ucb_norm4, d_norm4], axis=1)
top500_idx4 = np.argsort(S4)[::-1][:500]
is_pareto4  = is_non_dominated(objectives4[top500_idx4])
pareto_idx4 = top500_idx4[is_pareto4]

fig = plt.figure(figsize=(16, 11))
fig.suptitle("Iteration 4 — Acquisition Policy: Pareto Frontier & Objective Space (α=0.6)", 
#fig.suptitle("Iteration 3 — Acquisition Policy: Pareto Frontier & Objective Space",
             fontsize=14, fontweight="bold", y=0.99)

gs   = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)
ax_s = fig.add_subplot(gs[0, :])
ax_o = fig.add_subplot(gs[1, 0])
ax_p = fig.add_subplot(gs[1, 1])

# ─── Panel 1: Acquisition score across augmented pool ─────────────────
ax_s.scatter(np.arange(len(S4)), S4,
             c=S4, cmap="viridis", s=2, alpha=0.3, rasterized=True)
ax_s.scatter(top20_pos4, S4[top20_pos4],
             c="red", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_s.scatter(pareto_idx4, S4[pareto_idx4],
             c="gold", s=40, zorder=4, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated (top 500)")
ax_s.set_xlabel("Pool index")
ax_s.set_ylabel("Acquisition score S(x)")
ax_s.set_title("Acquisition Score Across Augmented Pool")
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.2)

# ─── Panel 2: Objective space with Pareto frontier ────────────────────
ax_o.scatter(ucb_norm4, d_norm4,
             c="steelblue", s=3, alpha=0.15, rasterized=True, label="All points")
ax_o.scatter(ucb_norm4[pareto_idx4], d_norm4[pareto_idx4],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.6, marker="*", label="Pareto-non-dominated")
ax_o.scatter(ucb_norm4[top20_pos4], d_norm4[top20_pos4],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
pareto_pts4 = np.stack([ucb_norm4[pareto_idx4], d_norm4[pareto_idx4]], axis=1)
sorted_p4   = pareto_pts4[np.argsort(pareto_pts4[:, 0])]
ax_o.step(sorted_p4[:, 0], sorted_p4[:, 1], where="post",
          color="gold", lw=2, ls="--", alpha=0.85, zorder=4, label="Pareto front")
ax_o.set_xlabel("UCB$_{norm}$(x) — Scientific Discovery")
ax_o.set_ylabel("$d_{norm}$(x) — Community Engagement")
ax_o.set_title("Objective Space with Pareto Frontier")
ax_o.legend(fontsize=9)
ax_o.grid(True, alpha=0.2)

# ─── Panel 3: PCA space ───────────────────────────────────────────────
sc = ax_p.scatter(X_pca4[:, 0], X_pca4[:, 1],
                  c=S4, cmap="viridis", s=3, alpha=0.3, rasterized=True)
ax_p.scatter(X_pca4[pareto_idx4, 0], X_pca4[pareto_idx4, 1],
             c="gold", s=60, zorder=5, edgecolors="black",
             linewidths=0.5, marker="*", label="Pareto-non-dominated")
ax_p.scatter(X_pca4[top20_pos4, 0], X_pca4[top20_pos4, 1],
             c="red", s=60, zorder=6, edgecolors="black",
             linewidths=0.6, label="Top 20 selected")
ax_p.set_xlabel("PC1")
ax_p.set_ylabel("PC2")
ax_p.set_title("PCA Feature Space\n(colour = acquisition score)")
ax_p.legend(fontsize=9)
ax_p.grid(True, alpha=0.2)
plt.colorbar(sc, ax=ax_p, label="S(x)")

plt.show()

Good distribution was maintained in iteration 4, and we're approaching the Pareto frontier again from iteration 3. 

?The EHVI term successfully reintroduced non-dominated selection, and the region diversity term kept the PCA feature space coverage broad. This is the most balanced result across all iterations.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_tr_eval4, X_te_eval4, y_tr_eval4, y_te_eval4 = train_test_split(
    X_train_aug4, y_train_aug4, test_size=0.2, random_state=1003
)

gp_eval4 = GaussianProcessRegressor(
    kernel=Matern(nu=2.5, length_scale=1.0, length_scale_bounds=(1e-2, 1e2)),
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=1003,
)
gp_eval4.fit(X_tr_eval4, y_tr_eval4)
mu_test4, sigma_test4 = gp_eval4.predict(X_te_eval4, return_std=True)

rmse4     = np.sqrt(mean_squared_error(y_te_eval4, mu_test4))
r2_4      = 1 - np.sum((y_te_eval4 - mu_test4)**2) / np.sum((y_te_eval4 - y_te_eval4.mean())**2)
mean_std4 = sigma_test4.mean()

print(f"Test RMSE : {rmse4:.4f}")
print(f"Test R²   : {r2_4:.4f}")
print(f"Mean σ    : {mean_std4:.4f}")

fig = plt.figure(figsize=(18, 11))
fig.suptitle("Iteration 4 — Surrogate Performance, Uncertainty & Acquisition Policy (α=0.6)",
#fig.suptitle("Iteration 3 — Surrogate Performance, Uncertainty & Acquisition Policy",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.35)

ax_fit = fig.add_subplot(gs[0, :2])
ax_res = fig.add_subplot(gs[0, 2:])
ax_ucb = fig.add_subplot(gs[1, 0])
ax_std = fig.add_subplot(gs[1, 1])
ax_div = fig.add_subplot(gs[1, 2])
ax_s   = fig.add_subplot(gs[1, 3])

# ─── Panel 1: Predicted vs actual ─────────────────────────────────────
ax_fit.scatter(y_te_eval4, mu_test4, c=sigma_test4, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
lims = [min(y_te_eval4.min(), mu_test4.min()), max(y_te_eval4.max(), mu_test4.max())]
ax_fit.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
ax_fit.set_xlabel("True y (scaled)")
ax_fit.set_ylabel("Predicted μ (scaled)")
ax_fit.set_title(f"Predicted vs Actual\nRMSE={rmse4:.4f}  R²={r2_4:.4f}")
ax_fit.legend(fontsize=9)
ax_fit.grid(True, alpha=0.25)
sm = plt.cm.ScalarMappable(cmap="coolwarm",
     norm=plt.Normalize(sigma_test4.min(), sigma_test4.max()))
plt.colorbar(sm, ax=ax_fit, label="σ (test points)")

# ─── Panel 2: Residuals ────────────────────────────────────────────────
residuals4 = y_te_eval4 - mu_test4
ax_res.scatter(mu_test4, residuals4, c=sigma_test4, cmap="coolwarm",
               s=8, alpha=0.5, rasterized=True)
ax_res.axhline(0, color="k", lw=1.5, ls="--")
ax_res.fill_between(
    np.linspace(mu_test4.min(), mu_test4.max(), 100),
    -2 * sigma_test4.mean(), 2 * sigma_test4.mean(),
    alpha=0.1, color="gray", label="±2σ (mean)"
)
ax_res.set_xlabel("Predicted μ")
ax_res.set_ylabel("Residual (true − predicted)")
ax_res.set_title(f"Residuals\nMean σ = {mean_std4:.4f}")
ax_res.legend(fontsize=9)
ax_res.grid(True, alpha=0.25)

# ─── Helper ───────────────────────────────────────────────────────────
def dist_panel(ax, values, selected_values, xlabel, title, color):
    ax.hist(values, bins=60, color=color, alpha=0.45, label="All pool points")
    ax.hist(selected_values, bins=12, color="red", alpha=0.85, label="Top 20 selected")
    ax.axvline(selected_values.mean(), color="darkred", lw=1.5,
               ls="--", label=f"Selected mean={selected_values.mean():.3f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

# ─── Panels 3–6 ───────────────────────────────────────────────────────
dist_panel(ax_ucb, ucb_norm4, ucb_norm4[top20_pos4],
           "UCB$_{norm}$(x)", "Scientific Discovery\nUCB distribution",       "#3498db")
dist_panel(ax_std, sigma4,    sigma4[top20_pos4],
           "σ(x)",            "Model Uncertainty\nσ distribution",             "#9b59b6")
dist_panel(ax_div, d_norm4,   d_norm4[top20_pos4],
           "$d_{norm}$(x)",   "Community Engagement\nDiversity distribution",  "#2ecc71")
dist_panel(ax_s,   S4,        S4[top20_pos4],
           "S(x)",            "Combined Acquisition\nScore S(x)",              "#e67e22")

plt.show()


In [ ]:
# ── Save full results for future reference ─────────────────────────────
results_iter4_save = results_iter4.copy()

feat_rename = {features[i]: f"x{i+1}" for i in range(11)}
results_iter4_save = results_iter4_save.rename(columns=feat_rename)
results_iter4_save["sis_id"] = "ansarir5"

save_cols = (
    ["sis_id", "Pool_index"]
    + [f"x{i}" for i in range(1, 12)]
    + ["y", "score", "ucb_norm", "d_norm", #"cs_norm",
       #"coverage_norm",
         "y_pred", "y_uncert", "scientist_id", "community_score"]
)

results_iter4_save[save_cols].to_csv("iteration4_results_full.csv", index=False)
print("Saved iteration4_results_full.csv")

# ── Save submission CSV ────────────────────────────────────────────────
def make_explanation_iter4(row):
    feat_cols = [f"x{i}" for i in range(1, 12)]
    top_feat  = row[feat_cols].abs().idxmax()
    n_scientists = len(row["scientist_id"].split(","))
    return (
        f"Selected via multi-objective acquisition (S={row['score']:.3f}) "
        f"balancing scientific discovery (UCB: mu={row['y_pred']:.3f}, sigma={row['y_uncert']:.3f}) "
        f"and community engagement (diversity={row['d_norm']:.3f}, "
        f"community_score={row['community_score']:.3f}, "
        f"proposed by {n_scientists} scientist(s): {row['scientist_id']}). "
        f"Most dominant feature: {top_feat}={row[top_feat]:.3f}."
    )

submission_iter4 = results_iter4_save.copy()
submission_iter4["explanation"] = submission_iter4.apply(make_explanation_iter3, axis=1)

feat_cols = [f"x{i}" for i in range(1, 12)]
submission_iter4 = submission_iter4[
    ["sis_id", "Pool_index"] + feat_cols + ["y", "score", "explanation"]
]

submission_iter4 = submission_iter4.sort_values("score", ascending=False).reset_index(drop=True)

print(submission_iter4[["sis_id", "Pool_index", "y", "score"]].to_string(index=False))

submission_iter4.to_csv("iteration4_submission.csv", index=False)
print("\nSaved iteration4_submission.csv")

In [ ]:
submission_iter4

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# ── Proposals per scientist ────────────────────────────────────────────
for sci in sorted(scientists):
    mask    = proposals_full["scientist_id"] == sci
    indices = proposals_full.loc[mask, "Pool_index"].values
    ax.scatter(
        indices, [sci] * len(indices),
        color=sci_color_map[sci], s=30,
        edgecolors="black", linewidths=0.4, alpha=0.8
    )

# ── Selected points overlay ────────────────────────────────────────────
for _, row in results_iter4.iterrows():
    for sci in row["scientist_id"].split(","):
        ax.scatter(
            row["Pool_index"], sci.strip(),
            color="red", s=120, marker="*",
            zorder=6, edgecolors="black", linewidths=0.8
        )

ax.scatter([], [], color="red", s=120, marker="*",
           edgecolors="black", linewidths=0.8, label="Selected")
ax.legend(fontsize=9)
ax.set_xlabel("Pool_index")
ax.set_title("Iteration 4 — Proposal Regions per Scientist (★ = selected)")
ax.tick_params(axis="y", labelsize=8)
ax.grid(True, alpha=0.2, axis="x")

plt.tight_layout()
plt.show()

This plot aims to see which scientists proposals were selected in the end with reference to the general pool index range. 

It is primarily to see if the happiest scientists are actually the ones whose proposals were accepted. 

## Iteration 5

In [ ]:
# ── Parse happiness scores from text file ──────────────────────────────
happiness_raw4 = open("Proposals_iter4/happiness_results.txt").read()

happiness4 = {}
lines = happiness_raw4.strip().split("\n")
current_proposal = None
for line in lines:
    line = line.strip()
    if line.startswith("Proposal file:"):
        current_proposal = line.split("Proposal file:")[-1].strip().replace(".csv", "")
    elif line.startswith("Happiness with samples selected"):
        score = float(line.split(":")[-1].strip())
        happiness4[current_proposal] = score

print("Parsed happiness scores:")
for k, v in sorted(happiness4.items()):
    print(f"  {k}: {v:.4f}")

In [ ]:
#comparing iteration2_submission and happiness_results

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd


# ── Map happiness back to each selected point ──────────────────────────
# Points proposed by multiple scientists get the mean happiness
def mean_happiness(scientist_id_str):
    scientists = scientist_id_str.split(",")
    return np.mean([happiness4[s.strip()] for s in scientists])

results_iter4["happiness"] = results_iter4["scientist_id"].apply(mean_happiness)


# ── Figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Iteration 4 — Top 20 Selected Points vs Community Happiness",
             fontsize=14, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

ax_bar  = fig.add_subplot(gs[0, :2])   # happiness per selected point
ax_sci  = fig.add_subplot(gs[0, 2])    # mean happiness per scientist
ax_sco  = fig.add_subplot(gs[1, 0])    # happiness vs acquisition score
ax_ucb  = fig.add_subplot(gs[1, 1])    # happiness vs UCB
ax_div  = fig.add_subplot(gs[1, 2])    # happiness vs diversity

# ─── Panel 1: Happiness per selected point ────────────────────────────
colors = plt.cm.RdYlGn(
    (results_iter4["happiness"] - 1) / 4
)
bars = ax_bar.bar(
    range(len(results_iter4)),
    results_iter4["happiness"],
    color=colors, edgecolor="black", linewidth=0.6
)
ax_bar.axhline(results_iter4["happiness"].mean(), color="black",
               lw=1.5, ls="--",
               label=f"Mean happiness = {results_iter4['happiness'].mean():.2f}")
ax_bar.set_xticks(range(len(results_iter4)))
ax_bar.set_xticklabels(results_iter4["Pool_index"], rotation=45, ha="right", fontsize=8)
ax_bar.set_xlabel("Pool_index")
ax_bar.set_ylabel("Happiness score")
ax_bar.set_title("Happiness Score per Selected Point\n(colour: red=low, green=high)")
ax_bar.set_ylim(0, 5.5)
ax_bar.legend(fontsize=9)
ax_bar.grid(True, alpha=0.2, axis="y")

# ─── Panel 2: Mean happiness per scientist ────────────────────────────
sci_df = pd.DataFrame(happiness4.items(), columns=["scientist_id", "happiness"])
sci_df = sci_df.sort_values("happiness", ascending=True)
bar_colors = plt.cm.RdYlGn((sci_df["happiness"] - 1) / 4)
ax_sci.barh(sci_df["scientist_id"], sci_df["happiness"],
            color=bar_colors, edgecolor="black", linewidth=0.6)
ax_sci.axvline(sci_df["happiness"].mean(), color="black", lw=1.5, ls="--",
               label=f"Mean = {sci_df['happiness'].mean():.2f}")
ax_sci.set_xlabel("Happiness score")
ax_sci.set_title("Happiness per Scientist")
ax_sci.set_xlim(0, 5.5)
ax_sci.legend(fontsize=9)
ax_sci.grid(True, alpha=0.2, axis="x")

# ─── Helper: scatter with correlation ─────────────────────────────────
def scatter_corr(ax, x, y, xlabel, title, color):
    ax.scatter(x, y, c=color, s=60, edgecolors="black", linewidths=0.6, alpha=0.8)
    # correlation
    r = np.corrcoef(x, y)[0, 1]
    m, b = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, m * xline + b, "k--", lw=1.5, label=f"r = {r:.2f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Happiness")
    ax.set_title(title)
    ax.set_ylim(0, 5.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

# ─── Panels 3–5: Happiness vs score components ────────────────────────
scatter_corr(ax_sco, results_iter4["score"],    results_iter4["happiness"],
             "Acquisition score S(x)",  "Happiness vs Acquisition Score", "#e67e22")
scatter_corr(ax_ucb, results_iter4["ucb_norm"], results_iter4["happiness"],
             "UCB$_{norm}$(x)",         "Happiness vs Scientific Discovery", "#3498db")
scatter_corr(ax_div, results_iter4["d_norm"],   results_iter4["happiness"],
             "$d_{norm}$(x)",           "Happiness vs Diversity", "#2ecc71")

plt.show()

- We can see that there was no scientist who was particularly unhappy with this iteration. 

- The mean happiness went down from iteration 3. 

- The acquisiton policy was able to capture the correlation between happiness and diversity again. The scientific discovery correlation is still lacking. 

- There was a general positive trend of scientists whose proposals were selected and their happiness. Scientist 5 only had 1 point selected and scientist 9 had none but they were still in top 3 scientists happy with this iteration. Scientist 13 had 3 points selected but was the least satisfied. This suggests a further issue of what the scientists are actually expecting from their points and they may not be interested in the feature space as much as they are interested in the results. 

## Assessing your model

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor

# ── Fit a fast RF surrogate to get feature importances ────────────────
# Use the full augmented training set from the last iteration
rf = RandomForestRegressor(n_estimators=200, random_state=1003, n_jobs=-1)
rf.fit(X_train_aug4, y_train_aug4)

# ── 1. Scientific Discovery importance ────────────────────────────────
# Target: UCB_norm — which features drive high predicted y + uncertainty
ucb_all, mu_all, sigma_all = objective_scientific_discovery(
    model_iter4.X_scaled_, model_iter4.gp_, kappa=model_iter4.kappa
)
ucb_norm_all = (ucb_all - ucb_all.min()) / (ucb_all.max() - ucb_all.min())

rf_ucb = RandomForestRegressor(n_estimators=200, random_state=1003, n_jobs=-1)
rf_ucb.fit(model_iter4.X_scaled_, ucb_norm_all)
importance_sci = rf_ucb.feature_importances_

# ── 2. Community Engagement importance ────────────────────────────────
# Target: d_norm — which features drive feature-space diversity
d_all, _ = objective_community_engagement(
    model_iter4.X_scaled_,
    n_components=model_iter4.n_components,
    k=model_iter4.k_neighbors
)
d_norm_all = (d_all - d_all.min()) / (d_all.max() - d_all.min())

rf_div = RandomForestRegressor(n_estimators=200, random_state=1003, n_jobs=-1)
rf_div.fit(model_iter4.X_scaled_, d_norm_all)
importance_eng = rf_div.feature_importances_

# ── 3. Combined acquisition score importance ──────────────────────────
S_all, _, _ = model_iter4.score()
rf_s = RandomForestRegressor(n_estimators=200, random_state=1003, n_jobs=-1)
rf_s.fit(model_iter4.X_scaled_, S_all)
importance_combined = rf_s.feature_importances_

feat_labels = [f"x{i+1}\n({fn})" for i, fn in enumerate([
    "Perimeter\nratio", "c4\nbase", "c8\nbase",
    "c4\ntop", "c8\ntop", "twist\nlinear",
    "twist\namplitude", "twist\ncycles",
    "height", "mass", "thickness"
])]

# ── Figure ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle("Feature Importance Analysis\nScientific Discovery vs Community Engagement",
             fontsize=14, fontweight="bold", y=1.01)

def importance_bar(ax, importances, title, color, ylabel="Importance"):
    order  = np.argsort(importances)[::-1]
    colors = [color if i == order[0] else "#aec6cf" for i in range(len(importances))]
    bars   = ax.bar(range(len(importances)), importances[order],
                    color=[colors[i] for i in np.argsort(order)],
                    edgecolor="black", linewidth=0.6)
    # Highlight top feature
    top_bar = np.argmax(importances)
    ax.bar(top_bar, importances[top_bar], color=color,
           edgecolor="black", linewidth=0.8,
           label=f"Most important: {feat_labels[top_bar].split(chr(10))[0]}")
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels(feat_labels, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis="y")
    ax.set_xlim(-0.5, len(importances) - 0.5)

importance_bar(axes[0], importance_sci,
               "Scientific Discovery — Features driving UCB$_{norm}$(x)\n"
               "(high predicted energy absorption + model uncertainty)",
               "#3498db")

importance_bar(axes[1], importance_eng,
               "Community Engagement — Features driving $d_{norm}$(x)\n"
               "(feature-space diversity, underexplored geometry regions)",
               "#2ecc71")

# ── Panel 3: side-by-side comparison ──────────────────────────────────
ax = axes[2]
x      = np.arange(len(feat_labels))
width  = 0.25
ax.bar(x - width, importance_sci,      width, label="Scientific Discovery (UCB)",
       color="#3498db", edgecolor="black", linewidth=0.6, alpha=0.85)
ax.bar(x,          importance_eng,      width, label="Community Engagement (diversity)",
       color="#2ecc71", edgecolor="black", linewidth=0.6, alpha=0.85)
ax.bar(x + width,  importance_combined, width, label="Combined S(x)",
       color="#e67e22", edgecolor="black", linewidth=0.6, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(feat_labels, fontsize=9)
ax.set_ylabel("Feature importance")
ax.set_title("Feature Importance Comparison\nScientific Discovery vs Community Engagement vs Combined S(x)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.savefig("feature_importance_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


The feature importance analysis finds that: 

- X9 (height) and X7 (twist amplitude) maximized scientific discovery
- X1 (perimeter ratio) and X9 (height) maximized community engagement 
- Overall, the acquisiton score was largely dictated by the height (x9) which was noted in the initial EDA and ANOVA analysis as a significant feature. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

iterations = [1, 2, 3, 4]
rmse_vals  = [rmse, rmse2, rmse3, rmse4]
r2_vals    = [r2, r2_2, r2_3, r2_4]
std_vals   = [mean_std, mean_std2, mean_std3, mean_std4]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Surrogate Model Performance Across Iterations", fontweight="bold")

for ax, vals, label, color in zip(
    axes,
    [rmse_vals, r2_vals, std_vals],
    ["Test RMSE", "Test R²", "Mean σ"],
    ["#e74c3c", "#2ecc71", "#9b59b6"]
):
    ax.plot(iterations, vals, marker="o", color=color, lw=2)
    ax.set_xlabel("Iteration")
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_xticks(iterations)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The errors and uncertainties of the model decreased with each iteration and the R2 increased.

This represents the improvement of the model with active learning, that the model is able to learn more with more experiments. 

The R2 was still quite low and this is because throughout the project greater emphasis was placed on acquisition policy rather than surrogate model training. 

This is a key limitation in this exploration, especially since the UCB is directly calculated from the gaussian process surrogate model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Happiness scores per iteration ─────────────────────────────────────
happiness_iter2 = happiness2   # dict: {scientist_id: score}
happiness_iter3 = happiness3
happiness_iter4 = happiness4

iterations = [2, 3, 4]
happiness_by_iter = [happiness_iter2, happiness_iter3, happiness_iter4]

# ── Build a tidy dataframe ─────────────────────────────────────────────
all_scientists = sorted(set(happiness_iter2) | set(happiness_iter3) | set(happiness_iter4))

records = []
for it, hdict in zip(iterations, happiness_by_iter):
    for sci in all_scientists:
        records.append({
            "iteration":    it,
            "scientist_id": sci,
            "happiness":    hdict.get(sci, np.nan)
        })

df = pd.DataFrame(records)
mean_by_iter = df.groupby("iteration")["happiness"].mean()

# ── Plot (single graph) ────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.title("Mean Community Happiness Across Iterations", fontsize=14, fontweight="bold")

plt.plot(
    mean_by_iter.index, mean_by_iter.values,
    marker="o", color="#e67e22", lw=2.5, label="Mean Happiness"
)

# Annotate points
for it, val in zip(mean_by_iter.index, mean_by_iter.values):
    plt.annotate(f"{val:.2f}", xy=(it, val),
                 xytext=(0, 10), textcoords="offset points",
                 ha="center", fontsize=10)

plt.xlabel("Iteration")
plt.ylabel("Mean happiness score")
plt.xticks(iterations)
plt.ylim(0, 5.5)
plt.grid(True, alpha=0.25)
plt.legend(fontsize=9)

plt.tight_layout()
plt.show()


Commmunity happiness was the best in iteration 3 but did not vary too much throughout the iterations. 

This suggests that the emphasis on community engagement terms were not the most impactful, rather a greater understanding of scientific discovery and how it is expressed in the acquisition policy influences community engagement relative to pure community engagement terms. 

## Summary 

Over the iterations, the acquisition policy for the project varied with a combination of scientific discovery and community engagement objectives as follows:

1. Utilized UCB and dnorm to select non-dominated points on the pareto frontier that were well-captured in the PCA feature space.
2. Introduced community scientist norm and coverage norm to prefer proposals that were proposed by different scientists. 
3. These terms were reweighted to emphasize scientific discovery because community happiness was not proportional to the intent with the terms.
4. Discovery emphasis was well received and further emphasized with EHVI. 

The primary limitation in this project was the inadequate training of the surrogate model. This impacted R2 as well as the UCB in the acquisition policy. In the future, greater emphasis should be placed on surrogate model training. 

Finally, iteration 3 returned the highest level of happiness from the community. But the deeper finding is that happiness and scientific discovery were structurally in tension and that's not a problem to solve by reweighting. There were scientists who had several points selected yet they were the most unhappy from the community. Greater community engagement and awareness would enable understanding of their personal objectives and perhaps split into subgroup such as academia vs industry to better target what they are looking for from the results. 

